# 📉 27 · Linear regression: the whole course in one model
### The regression problem, its matrix form, least squares and the normal equations, projection geometry and rank, the Gaussian noise model and maximum likelihood for β and σ², standard errors, t-tests, confidence and prediction intervals, multiple regression on the insulin study, a 0/1 predictor as a t-test, ridge regression as MAP — and a map of where every part of the course appears

**BiologyToMath** · Part G — Statistical inference in practice · Module 27 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/27_linear_regression_review.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, `scipy.optimize` — to check hand-written formulas), pandas, Matplotlib |
| **Prerequisites** | All earlier modules — this is the course review. Most used: [Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb), [Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb), [Module 11 · 🧱 Linear independence, span, basis and rank](11_independence_basis_rank.ipynb), [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb), [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb), [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb), [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb), [Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb) and [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb). |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Review via linear regression: independent and dependent variables, slope and intercept, biological examples of regression; the matrix form $\mathbf y = X\boldsymbol\beta$ with an augmented column of ones; least squares $\min\|\mathbf y - X\boldsymbol\beta\|_2^2$, expansion, derivative and the normal equations, $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf y$; the probabilistic model $p(y_i\mid x_i) = \mathcal N(\beta_0+\beta_s x_i,\sigma^2)$; maximum likelihood for $\boldsymbol\beta$ (= least squares) and for $\sigma^2$; the predictive distribution $p(\tilde y\mid\tilde x)$ — plus how logic, sets, vectors, matrices, linear systems, rank, eigenvalues, SVD, calculus, probability, estimation, MAP and hypothesis tests all appear in this one model |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. **All data in this notebook are SIMULATED by code**: the infection experiment (`simulate_tissue_expression`), the clinical insulin study (`simulate_insulin_study`, with known true coefficients), Hardy–Weinberg genotypes (`simulate_genotypes`) and small simulated regression data sets. Relationships between simulated genes are associations created by the simulator (both genes respond to infection), not measured regulatory mechanisms; the clinical coefficients are illustrative, not medical reference values.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Formulate** a biological regression problem — which variable is $x$, which is $y$, what the slope and intercept
   mean — and **write** it in matrix form $\mathbf y = X\boldsymbol\beta$ with an augmented column of ones.
2. **Derive**, line by line, the normal equations $X^\top X\boldsymbol\beta = X^\top\mathbf y$ from
   $\min\|\mathbf y - X\boldsymbol\beta\|_2^2$, and **explain** when $X^\top X$ is not invertible (rank, collinearity).
3. **Interpret** least squares geometrically as an orthogonal projection onto the column space of $X$, and **verify**
   the orthogonality, idempotence and trace properties numerically.
4. **Derive** maximum-likelihood estimates of $\boldsymbol\beta$ (= least squares) and $\sigma^2$ (= RSS/$n$) from the
   Gaussian noise model, and **explain** why RSS/$(n-p)$ is used in practice.
5. **Compute** standard errors, $t$-tests, confidence intervals, $R^2$ and prediction intervals for regression
   coefficients by hand, and **check** them against SciPy.
6. **Fit** a multiple regression to the insulin study three ways (normal equations, `lstsq`, gradient descent),
   **diagnose** its residuals and **explain** omitted-variable bias.
7. **Show** that regression on a 0/1 predictor is the two-sample $t$-test, and that MAP estimation with a Gaussian prior
   is ridge regression, which works even when there are more genes than samples.
8. **Map** every part of 02-680 — logic, sets, vectors, matrices, linear systems, rank, eigenvalues, SVD, calculus,
   probability, expectation, CLT, MLE, MAP and tests — onto a place where it is used in linear regression.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the expression and insulin data | 5 min |
| 1 · The regression problem | $x$ and $y$, slope and intercept, biological examples, a first scatter plot | 10 min |
| 2 · Matrix form | the augmented matrix $X = [\mathbf x, \mathbf 1]$; an overdetermined system | 10 min |
| 3 · Least squares and the normal equations | expand, differentiate, solve; four ways to compute $\hat{\boldsymbol\beta}$ | 25 min |
| 4 · Geometry: projection, rank and collinearity | the hat matrix, orthogonal residuals, a 3-D picture, singular $X^\top X$ | 20 min |
| 5 · The probabilistic model and maximum likelihood | Gaussian noise, MLE of $\boldsymbol\beta$ and $\sigma^2$, bias of RSS/$n$ | 25 min |
| 6 · Inference and prediction | standard errors, $t$-tests, CIs, $R^2$, coverage, prediction intervals | 20 min |
| 7 · Multiple regression: the insulin study | three solvers, a coefficient table, residual diagnostics, omitted variables | 25 min |
| 8 · Two special cases | a 0/1 predictor = the $t$-test; a GWAS scan with BH | 15 min |
| 9 · MAP = ridge regression | Gaussian prior, $(X^\top X+\lambda I)^{-1}X^\top\mathbf y$, more genes than samples | 15 min |
| 10 · The whole course in one model | the map, and a cell that checks a dozen course results on one fit | 15 min |
| Wrap-up | takeaways, cumulative checkpoint, questions, lab note | 15 min |

## 🧫 The biological question

Fasting insulin rises with body-mass index — but also with age and with fasting glucose, which itself rises with BMI.
***How much does each factor contribute, holding the others fixed, and how sure can we be?*** In a gene-expression
experiment, a transcription factor and its putative target go up together after infection. ***How much does the target
change per unit of the regulator, and is that a regulatory relationship or a shared response to the infection?***

Both questions are answered by the most used model in quantitative biology: **linear regression**. It is also the best
possible review of this course, because a single regression uses almost everything you have learned — tuples and vectors
to store the data, matrices and linear systems to solve for the coefficients, rank and eigenvalues to know when the
solution is unique, calculus to find the minimum, probability to model the noise, maximum likelihood and MAP to justify the
estimate, and hypothesis tests to judge it.

| Biological question | Mathematical tool | Where |
|---|---|---|
| How does a target gene's expression change with its regulator's? | simple regression, least squares | Sections 1–3 |
| When is the answer unique? What if two predictors carry the same information? | rank, projection, collinearity | Section 4 |
| Why least squares? How noisy is a measurement? | Gaussian model, MLE of $\boldsymbol\beta$ and $\sigma^2$ | Section 5 |
| Is the effect real? What will the next patient's value be? | $t$-tests, confidence and prediction intervals | Section 6 |
| BMI, age and glucose → insulin | multiple regression, diagnostics | Section 7 |
| Infected vs control; genotype → phenotype at many SNPs | 0/1 predictors, GWAS scan | Section 8 |
| 40 genes, 24 samples: predict a target gene | ridge regression = MAP | Section 9 |

> 🧪 **Simulated data, with a known truth.** The insulin study is generated from known coefficients
> (`INSULIN_TRUE_COEFFICIENTS`) and noise level (`INSULIN_TRUE_SIGMA`), so we can check every estimate against the truth.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `simulate_insulin_study` creates a SIMULATED clinical study (insulin ~ BMI + age + glucose + Gaussian noise) with known true coefficients `INSULIN_TRUE_COEFFICIENTS` and noise SD `INSULIN_TRUE_SIGMA`; `simulate_tissue_expression` is the course's SIMULATED infection experiment; `simulate_genotypes` draws SIMULATED 0/1/2 genotypes for the GWAS example. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools

import numpy as np
import pandas as pd


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}


def simulate_genotypes(n: int, p: float, inbreeding: float = 0.0, seed: int = 1866) -> np.ndarray:
    """SIMULATED genotypes of ``n`` diploid individuals at one SNP, coded 0/1/2 = number of copies of
    the allele with frequency ``p``.  With ``inbreeding`` F = 0 the genotype frequencies follow
    Hardy–Weinberg proportions ((1−p)², 2p(1−p), p²); F > 0 moves individuals from heterozygotes to
    homozygotes: P(1) = 2p(1−p)(1−F)."""
    rng = np.random.default_rng(seed)
    q = 1.0 - p
    probs = np.array([q * q + inbreeding * p * q, 2 * p * q * (1 - inbreeding), p * p + inbreeding * p * q])
    return rng.choice(3, size=n, p=probs / probs.sum())


INSULIN_TRUE_COEFFICIENTS = {"intercept": -4.0, "bmi": 0.55, "age": 0.04, "glucose": 0.06}


INSULIN_TRUE_SIGMA = 2.5


def simulate_insulin_study(n: int = 120, seed: int = 1921) -> pd.DataFrame:
    """SIMULATED clinical study: fasting insulin (µU/mL) of ``n`` adults as a linear function of
    BMI (kg/m²), age (years) and fasting glucose (mg/dL) plus Gaussian noise.  The true coefficients
    are ``INSULIN_TRUE_COEFFICIENTS`` and the noise SD is ``INSULIN_TRUE_SIGMA``.  Columns: bmi, age,
    glucose, insulin."""
    rng = np.random.default_rng(seed)
    bmi = rng.normal(27.0, 4.5, n).clip(17, 45)
    age = rng.uniform(20, 75, n)
    glucose = (88 + 1.1 * (bmi - 27) + 0.15 * (age - 45) + rng.normal(0, 9, n)).clip(65, 160)
    c = INSULIN_TRUE_COEFFICIENTS
    insulin = c["intercept"] + c["bmi"] * bmi + c["age"] * age + c["glucose"] * glucose \
        + rng.normal(0.0, INSULIN_TRUE_SIGMA, n)
    return pd.DataFrame({"bmi": bmi.round(1), "age": age.round(0), "glucose": glucose.round(0),
                         "insulin": insulin.round(2)})

print("🧰 toolkit loaded:", ", ".join(sorted(['HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', 'INSULIN_TRUE_COEFFICIENTS', 'INSULIN_TRUE_SIGMA', 'simulate_genotypes', 'simulate_insulin_study', 'simulate_tissue_expression'])))

In [ ]:
# ── 0.1 The two data sets of this module (SIMULATED) ──
expr = simulate_tissue_expression()
X_expr, genes, samples = expr["X"], expr["genes"], expr["samples"]
stat1 = X_expr[:, genes.index("STAT1")]          # regulator: a transcription factor of the interferon response
mx1 = X_expr[:, genes.index("MX1")]              # target: an interferon-stimulated antiviral gene
infected = (samples["condition"] == "infected").to_numpy()

insulin = simulate_insulin_study()               # 120 adults: bmi, age, glucose, insulin
print(f"expression: {X_expr.shape[0]} samples × {X_expr.shape[1]} genes; STAT1 and MX1 (log2) for the first 4 samples:")
print(pd.DataFrame({"sample": samples["sample"], "STAT1": stat1, "MX1": mx1}).head(4).round(3).to_string(index=False))
print(f"\ninsulin study: {insulin.shape[0]} adults")
print(insulin.head().to_string())
print("true coefficients:", INSULIN_TRUE_COEFFICIENTS, " true noise SD:", INSULIN_TRUE_SIGMA)

🔍 **What just happened?** We loaded two simulated data sets. The infection experiment gives, for each of the 24 tissue
samples, the log2 expression of the transcription factor *STAT1* and of the interferon-stimulated gene *MX1*. The insulin
study gives BMI (kg/m²), age (years), fasting glucose (mg/dL) and fasting insulin (µU/mL) for 120 adults; it was generated
as $\text{insulin} = -4 + 0.55\,\text{BMI} + 0.04\,\text{age} + 0.06\,\text{glucose} + \varepsilon$ with
$\varepsilon\sim\mathcal N(0, 2.5^2)$. Our job is to recover such numbers from data — and to know how well we did.

---
## 1 · The regression problem

> 📍 **Where we are.** [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb) met least squares as an optimisation problem and [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb) met maximum likelihood. This module
> puts them — and almost everything else in the course — together in one model.

The **linear regression** problem has four ingredients:

* $x$ — the **independent variable(s)**: what we control or use to predict (also *predictor*, *covariate*, *input*);
* $y$ — the **dependent variable(s)**: the measured values we want to explain (also *response*, *output*);
* $\beta_s$ — the **slope**: how much $y$ changes, on average, per unit increase of $x$;
* $\beta_0$ — the **intercept**: the average value of $y$ when $x = 0$.

The model says that $y \approx \beta_s x + \beta_0$ — a straight line — up to noise.

**Some examples of regression in biology:**

| Independent ($x$) | Dependent ($y$) |
|---|---|
| expression of genes of the immune response | asthma severity |
| expression of regulator genes | expression of target genes |
| clinical variables (BMI, age, glucose) | insulin level |
| genotype at a SNP (0/1/2 copies of an allele) | a quantitative trait, or a gene's expression (eQTL) |
| drug dose | response of a cell line |
| … | … |

⚠️ **Common confusion — "independent" here does not mean *statistically* independent** (Module 17). It only names the
variable we condition on. In observational biology we usually do **not** control $x$ (nobody assigns BMI), so a regression
measures an **association**, which may or may not be causal.

We observe $n$ samples, i.e. $n$ pairs $(x_i, y_i)$ — a set of 2-tuples (Modules 03–04) — and want the line that best
describes them. First, the picture.

In [ ]:
# ── 1.1 MX1 against STAT1 across the 24 samples (SIMULATED) ──
fig, ax = plt.subplots(figsize=(6.4, 3.8))
markers = {"spleen": "o", "kidney": "s", "liver": "^"}
for tissue, mk in markers.items():
    for cond, col in [("control", "tab:blue"), ("infected", "tab:red")]:
        m = ((samples["tissue"] == tissue) & (samples["condition"] == cond)).to_numpy()
        ax.scatter(stat1[m], mx1[m], marker=mk, color=col, s=36, alpha=0.8, label=f"{tissue}, {cond}")
ax.set_xlabel("STAT1 expression x (log2)"); ax.set_ylabel("MX1 expression y (log2)")
ax.set_title("regulator vs target across 24 samples (SIMULATED)")
ax.legend(fontsize=7, ncol=2, loc="upper left")
plt.tight_layout(); plt.show()
print(f"Pearson correlation r(STAT1, MX1) = {np.corrcoef(stat1, mx1)[0, 1]:.3f}")

🔍 **What just happened?** The 24 points rise from lower left to upper right: samples with more *STAT1* tend to have more
*MX1* (correlation $r = 0.77$, Module 19). The colours already hint at the reason: infected samples (red) sit high on both
axes, controls (blue) low on both. Keep that in mind — we return to it in Section 7. For now, we want the slope of the
cloud: how many log2 units of *MX1* go with one log2 unit of *STAT1*?

✏️ **Your turn.** For each row of the biology table, say what the slope would mean in words and in units. For example:
"BMI → insulin: $\beta_s$ is the average increase in fasting insulin, in µU/mL, per extra kg/m² of BMI."

---
## 2 · The matrix form

> 📍 **Where we are.** One equation per sample is clumsy. Vectors ([Module 07 · ➡️ Vectors: profiles, lengths and angles](07_vectors.ipynb)) and matrices ([Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb)) let us write all $n$
> equations at once — and then the whole of linear algebra is available.

Let $\mathbf x, \mathbf y\in\mathbb R^n$ hold the $n$ samples, so that $(x_i, y_i)$ belong together. The model for every
sample at once is

$$\mathbf y = \beta_s\,\mathbf x + \beta_0\,\mathbf 1, \qquad\text{i.e.}\qquad y_i = \beta_s x_i + \beta_0\quad (i = 1,\dots,n),$$

*In words:* the vector of responses is the slope times the vector of predictor values plus the intercept times the vector
of ones $\mathbf 1 = (1,\dots,1)^\top$ — a linear combination of two vectors (Module 11).

⚠️ **Shapes.** You may see this written "$y = x^T\beta_s + \beta_0$". With $\mathbf x\in\mathbb R^n$ a column vector and a
scalar slope, $\mathbf x^\top\beta_s$ is a *row* vector and adding the scalar $\beta_0$ is not defined; the consistent form
is $\beta_s\mathbf x + \beta_0\mathbf 1$ (or, one sample at a time, $y_i = \mathbf x_i^\top\boldsymbol\beta$ with
$\mathbf x_i = (x_i, 1)^\top$ — see below).

**The augmented matrix.** To simplify, define the $n\times2$ matrix $X$ whose first column is $\mathbf x$ and whose second
column is all ones, and the parameter vector $\boldsymbol\beta$:

$$X = \begin{bmatrix} x_1 & 1\\ x_2 & 1\\ \vdots & \vdots\\ x_n & 1\end{bmatrix},\qquad
\boldsymbol\beta = \begin{bmatrix}\beta_s\\ \beta_0\end{bmatrix}, \qquad\text{then}\qquad \mathbf y = X\boldsymbol\beta .$$

*In words:* row $i$ of $X$ times $\boldsymbol\beta$ is $x_i\beta_s + 1\cdot\beta_0$ — the prediction for sample $i$. The
column of ones is the trick that turns the intercept into an ordinary coefficient. With $J$ predictors, $X$ gets $J$
columns plus the column of ones — one row per sample, one column per feature, as everywhere in this course.

With real data, $\mathbf y = X\boldsymbol\beta$ has **no exact solution**: 24 equations, 2 unknowns, and the points are not
on a line. It is an **overdetermined** linear system ([Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb)).

In [ ]:
# ── 2.1 Build X = [x, 1] and check that y = Xβ has no exact solution ──
n = stat1.size
X1 = np.column_stack([stat1, np.ones(n)])                 # n × 2 augmented design matrix
print("first five rows of X:\n", X1[:5].round(3))
print("X has shape", X1.shape, "→ the system has", X1.shape[0], "equations in", X1.shape[1], "unknowns")
rank_X = np.linalg.matrix_rank(X1)
rank_Xy = np.linalg.matrix_rank(np.column_stack([X1, mx1]))
print(f"rank(X) = {rank_X}, rank([X | y]) = {rank_Xy}  →  "
      f"{'consistent' if rank_X == rank_Xy else 'inconsistent: no β solves all equations exactly'}")
beta_two = np.linalg.solve(X1[:2], mx1[:2])               # a line through the first two points only
print(f"the line through the first two points, β = {beta_two.round(3)}, misses the other points by up to "
      f"{np.max(np.abs(mx1 - X1 @ beta_two)):.2f} log2 units")

🔍 **What just happened?** $X$ is $24\times2$. Appending $\mathbf y$ as a third column raises the rank from 2 to 3 —
the consistency criterion behind the contradictory "0 = 6" rows of [Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb): $\mathbf y$ is not in the column space of $X$, so no $\boldsymbol\beta$ solves all 24
equations. A line forced through two of the points misses others badly. We need a *best compromise*.

### 📘 Lecture notes — the regression problem and its matrix form

- **Ingredients:** $x$ = independent variable(s) (what we control / predict from), $y$ = dependent variable(s) (the
  measured values), $\beta_s$ = slope, $\beta_0$ = intercept.
- **Biological examples:** immune-response gene expression → asthma severity; regulator gene expression → target gene
  expression; clinical variables → insulin level.
- **Data:** $\mathbf x, \mathbf y\in\mathbb R^n$, pairs $(x_i, y_i)$. Model $\mathbf y = \beta_s\mathbf x+\beta_0\mathbf 1$.
- **Augmented form:** $X = [\mathbf x\ \ \mathbf 1]$ ($n\times2$), $\boldsymbol\beta = (\beta_s,\beta_0)^\top$,
  $\mathbf y = X\boldsymbol\beta$; with $J$ predictors $X$ is $n\times(J+1)$.
- Real data make $\mathbf y = X\boldsymbol\beta$ **overdetermined and inconsistent** (rank$[X\mid\mathbf y]$ >
  rank $X$) → we minimise the error instead.

> 🎓 **Exam tip.** Write $X$ out with its column of ones and check the shapes: $X$ is $n\times2$, $\boldsymbol\beta$ is $2\times1$, so $X\boldsymbol\beta$ is $n\times1$ like $\mathbf y$.

---
## 3 · Least squares and the normal equations

> 📍 **Where we are.** [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb) derived the least-squares solution as a calculus exercise. Here we redo it in full,
> because every later section builds on the result.

We look for the $\boldsymbol\beta$ that minimises the **squared $\ell_2$ norm of the residual vector**
$\mathbf e = \mathbf y - X\boldsymbol\beta$ ([Module 07 · ➡️ Vectors: profiles, lengths and angles](07_vectors.ipynb)):

$$\hat{\boldsymbol\beta} = \underset{\boldsymbol\beta}{\arg\min}\ \|\mathbf y - X\boldsymbol\beta\|_2^2 = \underset{\boldsymbol\beta}{\arg\min}\ \sum_{i=1}^n\big(y_i - \beta_s x_i - \beta_0\big)^2 .$$

*In words:* choose the line that makes the sum of squared vertical distances between the points and the line as small as
possible (the **residual sum of squares**, RSS).

**Step 1 — expand.** Using $\|\mathbf v\|_2^2 = \mathbf v^\top\mathbf v$ and $(AB)^\top = B^\top A^\top$ ([Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb)):

$$\begin{aligned}
\|\mathbf y-X\boldsymbol\beta\|_2^2 &= (\mathbf y-X\boldsymbol\beta)^\top(\mathbf y-X\boldsymbol\beta)\\
&= (\mathbf y^\top-\boldsymbol\beta^\top X^\top)(\mathbf y-X\boldsymbol\beta)\\
&= \mathbf y^\top\mathbf y - \mathbf y^\top X\boldsymbol\beta - \boldsymbol\beta^\top X^\top\mathbf y + \boldsymbol\beta^\top X^\top X\boldsymbol\beta\\
&= \mathbf y^\top\mathbf y + \boldsymbol\beta^\top X^\top X\boldsymbol\beta - 2\boldsymbol\beta^\top X^\top\mathbf y .
\end{aligned}$$

*In words:* multiply out like $(a-b)^2$; the two middle terms are equal because each is a $1\times1$ number and a number
equals its own transpose: $\mathbf y^\top X\boldsymbol\beta = (\mathbf y^\top X\boldsymbol\beta)^\top = \boldsymbol\beta^\top X^\top\mathbf y$.

**Step 2 — differentiate and set to zero.** With the gradient identities of [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb),
$\nabla_{\boldsymbol\beta}(\boldsymbol\beta^\top\mathbf b) = \mathbf b$ and
$\nabla_{\boldsymbol\beta}(\boldsymbol\beta^\top A\boldsymbol\beta) = 2A\boldsymbol\beta$ for a **symmetric** $A$ (here
$A = X^\top X$, which is symmetric because $(X^\top X)^\top = X^\top X$); $\mathbf y^\top\mathbf y$ does not depend on
$\boldsymbol\beta$:

$$\begin{aligned}
\nabla_{\boldsymbol\beta}\big(\mathbf y^\top\mathbf y + \boldsymbol\beta^\top X^\top X\boldsymbol\beta - 2\boldsymbol\beta^\top X^\top\mathbf y\big) &= \mathbf 0\\
2X^\top X\boldsymbol\beta - 2X^\top\mathbf y &= \mathbf 0\\
X^\top X\,\boldsymbol\beta &= X^\top\mathbf y \qquad\text{(the normal equations)}\\
\hat{\boldsymbol\beta} &= (X^\top X)^{-1}X^\top\mathbf y \qquad\text{(if } X^\top X \text{ is invertible).}
\end{aligned}$$

*In words:* at the minimum the gradient vanishes; that gives a small square linear system (2×2 here), whose solution is
the least-squares estimate. **Is it a minimum?** The matrix of second derivatives (the Hessian) is $2X^\top X$, which is
positive semidefinite because $\mathbf v^\top X^\top X\mathbf v = \|X\mathbf v\|^2\ge0$ ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)): the RSS is a bowl, and
every critical point is a minimum.

⚠️ **Order of the factors.** $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf y$ — the inverse comes **first**.
"$X^\top\mathbf y\,(X^\top X)^{-1}$" multiplies a $2\times1$ vector by a $2\times2$ matrix, which is not even defined
(matrix multiplication is not commutative, [Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb)).

In [ ]:
# ── 3.1 The normal equations for MX1 ~ STAT1, by hand (2 × 2) and four ways by computer ──
XtX = X1.T @ X1
Xty = X1.T @ mx1
print("XᵀX =\n", XtX.round(3), "\nXᵀy =", Xty.round(3))
a, b, c, d = XtX[0, 0], XtX[0, 1], XtX[1, 0], XtX[1, 1]
det = a * d - b * c
XtX_inv = np.array([[d, -b], [-c, a]]) / det                       # the 2 × 2 inverse formula
beta_hand = XtX_inv @ Xty
print(f"det(XᵀX) = {det:.3f};  β̂ = (XᵀX)⁻¹Xᵀy = {beta_hand.round(4)}   (slope, intercept)")

print("np.linalg.solve(XᵀX, Xᵀy):", np.linalg.solve(XtX, Xty).round(4))
print("np.linalg.lstsq(X, y):     ", np.linalg.lstsq(X1, mx1, rcond=None)[0].round(4))
print("np.polyfit(x, y, 1):       ", np.polyfit(stat1, mx1, 1).round(4))
lr = stats.linregress(stat1, mx1)
print(f"scipy.stats.linregress:     slope {lr.slope:.4f}, intercept {lr.intercept:.4f}")
try:
    Xty.reshape(2, 1) @ XtX_inv                                   # the wrong order: (2×1)(2×2)
except ValueError as err:
    print("Xᵀy (XᵀX)⁻¹ →", type(err).__name__, "— shapes (2,1) and (2,2) do not match")

🔍 **What just happened?** $X^\top X$ collects three sums — $\sum x_i^2 = 1223.28$, $\sum x_i = 169.52$ (twice) and
$n = 24$ — and $X^\top\mathbf y = (\sum x_iy_i, \sum y_i) = (1311.36, 182.19)$. Inverting the $2\times2$ matrix with the
formula of [Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb) gives **$\hat\beta_s = 0.9448$, $\hat\beta_0 = 0.9176$**: each extra log2 unit of *STAT1* goes with
0.94 log2 units more *MX1*, on average. `solve`, `lstsq`, `polyfit` and `linregress` agree to every digit. The wrong order
of factors raises a shape error.

**The simple-regression formulas.** Writing the $2\times2$ normal equations out and dividing by $n$ gives
$\beta_s\overline{x^2} + \beta_0\bar x = \overline{xy}$ and $\beta_s\bar x + \beta_0 = \bar y$. The second equation says the
line passes through the point of means: $\hat\beta_0 = \bar y - \hat\beta_s\bar x$. Substituting into the first,

$$\hat\beta_s\big(\overline{x^2}-\bar x^2\big) = \overline{xy}-\bar x\,\bar y
\qquad\Longrightarrow\qquad
\hat\beta_s = \frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2} = \frac{\widehat{\operatorname{Cov}}(x,y)}{\widehat{\operatorname{Var}}(x)} = r\,\frac{s_y}{s_x}.$$

*In words:* the slope is the covariance of $x$ and $y$ divided by the variance of $x$ (Modules 19 and 21) — equivalently
the correlation rescaled from "SD units" to the units of $y$ per unit of $x$.

In [ ]:
# ── 3.2 Check the covariance formula, and look at the RSS bowl ──
cov_xy = np.cov(stat1, mx1)[0, 1]
var_x = np.var(stat1, ddof=1)
r_xy = np.corrcoef(stat1, mx1)[0, 1]
print(f"Cov/Var = {cov_xy:.4f}/{var_x:.4f} = {cov_xy / var_x:.4f};  r·s_y/s_x = {r_xy * mx1.std(ddof=1) / stat1.std(ddof=1):.4f}")
print(f"ȳ − β̂_s·x̄ = {mx1.mean():.4f} − {beta_hand[0]:.4f}·{stat1.mean():.4f} = {mx1.mean() - beta_hand[0] * stat1.mean():.4f}")


def rss(beta, X, y):
    """Residual sum of squares ‖y − Xβ‖² of a linear model."""
    resid = y - X @ beta
    return resid @ resid


bs = np.linspace(beta_hand[0] - 1, beta_hand[0] + 1, 121)
b0 = np.linspace(beta_hand[1] - 7, beta_hand[1] + 7, 121)
R = np.array([[rss(np.array([s_, i_]), X1, mx1) for s_ in bs] for i_ in b0])
fig, ax = plt.subplots(figsize=(5.6, 3.8))
cs = ax.contour(bs, b0, R, levels=np.quantile(R, [0.01, 0.03, 0.07, 0.15, 0.3, 0.5, 0.7]), cmap="viridis")
ax.clabel(cs, fontsize=6, fmt="%.0f")
ax.plot(*beta_hand, "r*", ms=12, label=f"β̂ = ({beta_hand[0]:.2f}, {beta_hand[1]:.2f})")
ax.set_xlabel("slope β_s"); ax.set_ylabel("intercept β_0"); ax.legend()
ax.set_title("RSS(β) for MX1 ~ STAT1: a long, narrow bowl")
plt.tight_layout(); plt.show()
eps = 1e-6
grad_fd = [(rss(beta_hand + eps * e_, X1, mx1) - rss(beta_hand - eps * e_, X1, mx1)) / (2 * eps) for e_ in np.eye(2)]
print("finite-difference gradient of RSS at β̂:", np.round(grad_fd, 6), "  formula 2XᵀXβ̂ − 2Xᵀy:",
      np.round(2 * XtX @ beta_hand - 2 * Xty, 10))

🔍 **What just happened?** The covariance formula, the correlation formula and the "line through the means" all reproduce
$\hat\beta_s = 0.9448$, $\hat\beta_0 = 0.9176$. The contour plot shows the RSS as a function of the two parameters: a
bowl with a single lowest point (red star). It is long and tilted because slope and intercept trade off — a steeper line
can be compensated by a lower intercept, since all $x$ values are far from 0 (around 7). At $\hat{\boldsymbol\beta}$ the
numerical gradient is zero up to rounding, as the normal equations require.

✏️ **Your turn.** Fit the line *within the spleen samples only* (8 samples) and within the liver samples. Are the slopes
the same as the overall 0.94? (Section 7 explains why they need not be.)

---
## 4 · Geometry: projection, rank and collinearity

> 📍 **Where we are.** The normal equations have a beautiful geometric meaning that uses vector spaces ([Module 10 · 🌌 Vector spaces and subspaces](10_vector_spaces.ipynb)), span and
> rank ([Module 11 · 🧱 Linear independence, span, basis and rank](11_independence_basis_rank.ipynb)) and eigenvalues ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)).

The fitted values $\hat{\mathbf y} = X\hat{\boldsymbol\beta}$ are a linear combination of the columns of $X$, so they lie
in the **column space** of $X$ (the span of $\mathbf x$ and $\mathbf 1$ — a 2-dimensional subspace of $\mathbb R^n$).
Least squares picks the point of that subspace **closest** to $\mathbf y$, i.e. the **orthogonal projection**. Rewrite the
normal equations:

$$X^\top X\hat{\boldsymbol\beta} = X^\top\mathbf y \iff X^\top(\mathbf y - X\hat{\boldsymbol\beta}) = \mathbf 0 \iff X^\top\mathbf e = \mathbf 0 .$$

*In words:* the residual vector is **orthogonal to every column of $X$** ([Module 07 · ➡️ Vectors: profiles, lengths and angles](07_vectors.ipynb): dot product zero). With the column of
ones this gives $\sum_i e_i = 0$ (residuals average to zero), and with the column $\mathbf x$, $\sum_i x_ie_i = 0$.

**The hat matrix.** Substituting $\hat{\boldsymbol\beta}$,

$$\hat{\mathbf y} = X(X^\top X)^{-1}X^\top\mathbf y = H\mathbf y, \qquad H = X(X^\top X)^{-1}X^\top \quad (n\times n).$$

*In words:* $H$ "puts the hat on $\mathbf y$". It is **symmetric** ($H^\top = H$) and **idempotent** ($H^2 = H$:
projecting twice changes nothing), its eigenvalues are 1 (directions in the column space) and 0 (directions orthogonal to
it), and its trace equals its rank, the number of columns $p$ of $X$. Because $\hat{\mathbf y}\perp\mathbf e$,
Pythagoras ([Module 07 · ➡️ Vectors: profiles, lengths and angles](07_vectors.ipynb)) gives $\|\mathbf y\|^2 = \|\hat{\mathbf y}\|^2 + \|\mathbf e\|^2$.

In [ ]:
# ── 4.1 Projection facts, checked numerically on MX1 ~ STAT1 ──
y_hat = X1 @ beta_hand
e = mx1 - y_hat
H = X1 @ np.linalg.inv(X1.T @ X1) @ X1.T
print("Xᵀe =", (X1.T @ e).round(12), "  → Σe_i =", round(e.sum(), 12), ", Σx_i e_i =", round(stat1 @ e, 12))
print("max |Hy − ŷ| =", np.abs(H @ mx1 - y_hat).max().round(12))
print("symmetric:", np.allclose(H, H.T), "  idempotent (H² = H):", np.allclose(H @ H, H))
eig_H = np.linalg.eigvalsh(H)
print(f"eigenvalues of H: {np.sum(np.isclose(eig_H, 1))} ones and {np.sum(np.isclose(eig_H, 0))} zeros; "
      f"trace(H) = {np.trace(H):.6f} = rank(X) = {np.linalg.matrix_rank(X1)}")
print(f"Pythagoras: ‖y‖² = {mx1 @ mx1:.4f}  vs  ‖ŷ‖² + ‖e‖² = {y_hat @ y_hat + e @ e:.4f}")

🔍 **What just happened?** Every statement checks out to about twelve decimals: $X^\top\mathbf e = \mathbf 0$ (so the
residuals sum to zero and are uncorrelated with *STAT1*), $H\mathbf y = \hat{\mathbf y}$, $H$ is symmetric and idempotent,
it has exactly **2 eigenvalues equal to 1 and 22 equal to 0**, its trace is 2 = rank($X$), and
$\|\mathbf y\|^2 = \|\hat{\mathbf y}\|^2+\|\mathbf e\|^2$. To *see* the projection we need a space we can draw — so take a
toy data set with only $n = 3$ samples, where $\mathbf y$ lives in $\mathbb R^3$ and the column space of $X$ is a plane.

In [ ]:
# ── 4.2 Least squares as a projection, drawn in ℝ³ (n = 3 toy samples) ──
x3 = np.array([1.0, 2.0, 3.0]); y3 = np.array([1.0, 3.0, 2.0])
X3 = np.column_stack([x3, np.ones(3)])
b3 = np.linalg.solve(X3.T @ X3, X3.T @ y3)
yh3, e3 = X3 @ b3, y3 - X3 @ b3
print(f"β̂ = {b3}, ŷ = {yh3}, e = {e3}, Xᵀe = {np.round(X3.T @ e3, 12)}")

fig = plt.figure(figsize=(10, 4.4))
views = [(25, 120, "looking onto the plane span{x, 1}"),
         (0, np.degrees(np.arctan(0.5)), "edge-on: the plane is a line and e ⊥ plane")]
for k, (elev, azim, title) in enumerate(views):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ax.set_proj_type("ortho")                                   # no perspective, so right angles look right
    u, v = np.meshgrid(np.linspace(-0.1, 1.0, 10), np.linspace(-0.3, 1.2, 10))
    plane = u[..., None] * x3 + v[..., None] * np.ones(3)      # all combinations u·x + v·1
    ax.plot_surface(plane[..., 0], plane[..., 1], plane[..., 2], alpha=0.2, color="tab:blue")
    for vec, col, lab in [(x3, "tab:blue", "column x"), (np.ones(3), "tab:cyan", "column 1"),
                          (y3, "k", "data y"), (yh3, "tab:green", "fit ŷ = Hy")]:
        ax.quiver(0, 0, 0, *vec, color=col, arrow_length_ratio=0.08, lw=2, label=lab)
    ax.plot(*zip(yh3, y3), color="tab:red", lw=2.5, ls="--", label="residual e = y − ŷ")
    for set_lim in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        set_lim(-0.5, 3.5)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(title, fontsize=9)
    if k == 0:
        ax.set_xlabel("sample 1"); ax.set_ylabel("sample 2"); ax.set_zlabel("sample 3")
    else:
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each axis is one *sample*, so a whole data set is one point (arrow) in $\mathbb R^3$. The blue
plane is the set of all $\beta_s\mathbf x + \beta_0\mathbf 1$ — every vector a straight line could produce. The data
vector $\mathbf y = (1, 3, 2)$ sticks out of the plane; the fit $\hat{\mathbf y} = (1.5, 2, 2.5)$ is the foot of the
perpendicular, and the red residual $\mathbf e = (-0.5, 1, -0.5)$ is orthogonal to the plane ($X^\top\mathbf e = \mathbf 0$).
The right panel looks along the plane (perpendicular to its normal vector $\mathbf x\times\mathbf 1 = (-1, 2, -1)$, which
happens to be parallel to $\mathbf e$): the plane collapses to a line and the right angle between it and the residual
is plain to see.
The same picture holds in $\mathbb R^{24}$ — we just cannot draw it.

### When $X^\top X$ is not invertible: collinear predictors

The formula $(X^\top X)^{-1}X^\top\mathbf y$ needs $X^\top X$ to be invertible, which holds **if and only if the columns
of $X$ are linearly independent**, i.e. rank $X = p$ (the number of columns) — [Module 11 · 🧱 Linear independence, span, basis and rank](11_independence_basis_rank.ipynb). (Reason:
$X^\top X\mathbf v = \mathbf 0 \Rightarrow \mathbf v^\top X^\top X\mathbf v = \|X\mathbf v\|^2 = 0 \Rightarrow X\mathbf v = \mathbf 0$,
so the two matrices have the same null space.) Biology produces dependent columns more often than you might think: the same
measurement in two units, a marker gene that is the sum of two others, more genes than samples. Let us add glucose in
**mmol/L** to the insulin model that already has glucose in **mg/dL** (1 mmol/L = 18.016 mg/dL for glucose).

In [ ]:
# ── 4.3 A perfectly collinear design: glucose in mg/dL and in mmol/L (SIMULATED insulin study) ──
y_ins = insulin["insulin"].to_numpy()
X_ins = np.column_stack([np.ones(len(insulin)), insulin[["bmi", "age", "glucose"]].to_numpy()])
X_dup = np.column_stack([X_ins, insulin["glucose"].to_numpy() / 18.016])       # the same information again
for name, M in [("1, bmi, age, glucose", X_ins), ("… + glucose in mmol/L", X_dup)]:
    G = M.T @ M
    print(f"{name:22s}: {M.shape[1]} columns, rank {np.linalg.matrix_rank(M)}, "
          f"smallest eigenvalue of XᵀX = {np.linalg.eigvalsh(G).min():.2e}, condition number = {np.linalg.cond(M):.1e}")

null_dir = np.array([0, 0, 0, 1.0, -18.016])                 # X_dup @ null_dir = glucose − 18.016·(glucose/18.016) = 0
print("‖X·v‖ for v = (0, 0, 0, 1, −18.016):", np.linalg.norm(X_dup @ null_dir).round(10), "→ v is in the null space")
b_min = np.linalg.lstsq(X_dup, y_ins, rcond=None)[0]
b_other = b_min + 0.05 * null_dir
print("lstsq (minimum-norm) solution:", b_min.round(4), " RSS =", round(rss(b_min, X_dup, y_ins), 4))
print("another solution, β + 0.05·v: ", b_other.round(4), " RSS =", round(rss(b_other, X_dup, y_ins), 4))

🔍 **What just happened?** With the duplicated glucose column, $X$ has 5 columns but rank 4: the smallest eigenvalue of
$X^\top X$ is zero up to rounding and the condition number explodes, so $X^\top X$ is singular. The vector
$\mathbf v = (0,0,0,1,-18.016)$ lies in the **null space** of $X$ ([Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb), [Module 10 · 🌌 Vector spaces and subspaces](10_vector_spaces.ipynb)): adding any multiple of it to
$\hat{\boldsymbol\beta}$ changes the coefficients but not the fit — both solutions printed have exactly the same RSS. The
data cannot say how to split the glucose effect between the two columns. `lstsq` returns the solution of smallest norm
(computed with the SVD of [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb)); ridge regression (Section 9) is another way out. Even the *original* design has a large
condition number (≈ 10³) because the columns have very different scales — which matters for gradient descent (Section 7).

### 📘 Lecture notes — least squares, the normal equations and their geometry

- **Objective:** minimise $\|\mathbf y-X\boldsymbol\beta\|_2^2$ (square of the $L_2$ norm of the residuals).
- **Expansion:** $(\mathbf y-X\boldsymbol\beta)^\top(\mathbf y-X\boldsymbol\beta) = (\mathbf y^\top-\boldsymbol\beta^\top X^\top)(\mathbf y-X\boldsymbol\beta)
  = \mathbf y^\top\mathbf y + \boldsymbol\beta^\top X^\top X\boldsymbol\beta - 2\boldsymbol\beta^\top X^\top\mathbf y$.
- **Critical point:** $\frac{d}{d\boldsymbol\beta}(\cdot) = 2X^\top X\boldsymbol\beta - 2X^\top\mathbf y = \mathbf 0$ ⇒
  **normal equations** $X^\top X\boldsymbol\beta = X^\top\mathbf y$ ⇒ $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf y$
  (inverse first). Hessian $2X^\top X\succeq0$ ⇒ minimum.
- **Simple regression:** $\hat\beta_s = \sum(x_i-\bar x)(y_i-\bar y)/\sum(x_i-\bar x)^2 = \text{Cov}/\text{Var}$,
  $\hat\beta_0 = \bar y-\hat\beta_s\bar x$. Example: *MX1* on *STAT1*: $\hat\beta_s = 0.945$, $\hat\beta_0 = 0.918$.
- **Geometry:** $\hat{\mathbf y} = H\mathbf y$, $H = X(X^\top X)^{-1}X^\top$ = orthogonal projection onto the column space;
  $X^\top\mathbf e = \mathbf 0$; $H$ symmetric, idempotent, eigenvalues 0/1, trace $=p$.
- **Uniqueness:** $X^\top X$ invertible ⇔ columns of $X$ independent ⇔ rank $X = p$. Collinear predictors (same variable
  twice, more features than samples) ⇒ infinitely many least-squares solutions (add any null-space vector).

> 🎓 **Exam tip.** Know both derivations of the normal equations — calculus (set the gradient to zero) and geometry (the residual is orthogonal to the column space) — and why $X^\top X$ and $X$ have the same null space.

---
## 5 · The probabilistic model and maximum likelihood

> 📍 **Where we are.** Least squares is a geometric recipe. Why *squares*, and how uncertain is the answer? A probability
> model ([Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb)) answers both, through maximum likelihood ([Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb)).

What if we want the probability of some $y_i$ given $x_i$, i.e. $p(y_i\mid x_i)$? Assuming the measurements come from some
natural process with many small independent disturbances (the CLT of [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) makes Gaussian noise a reasonable default),
we assume

$$p(y_i\mid x_i) = \mathcal N(\beta_0+\beta_s x_i,\ \sigma^2) \qquad\text{or equivalently}\qquad y_i = \beta_0+\beta_s x_i+\varepsilon_i,\quad \varepsilon_i\sim\mathcal N(0,\sigma^2)\ \text{i.i.d.}$$

*In words:* for each value of $x$ there is a bell curve of possible $y$ values, centred on the line, all with the same
width σ. Adding a constant to a Gaussian shifts its mean, which is why the two forms are the same statement. σ measures how
uncertain we are about any given output — the biological plus technical noise around the trend.

So, given the data $\mathcal D = \{(x_i,y_i)\}_{i=1}^n$, we can use **maximum likelihood to find $\boldsymbol\beta$** (and
σ). Write $\mathbf x_i^\top = (x_i, 1)$ for row $i$ of $X$, so that $\beta_0+\beta_sx_i = \mathbf x_i^\top\boldsymbol\beta$.

In [ ]:
# ── 5.1 What the model says: a bell curve of y around the line at every x ──
sigma_hat_mle = np.sqrt(e @ e / n)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.scatter(stat1, mx1, s=18, color="tab:blue", alpha=0.7, label="data")
xg = np.linspace(stat1.min() - 0.3, stat1.max() + 0.3, 100)
ax.plot(xg, beta_hand[1] + beta_hand[0] * xg, "k-", label="fitted line β̂₀ + β̂_s·x")
yg = np.linspace(mx1.min() - 1.5, mx1.max() + 1.5, 200)
for x0 in [5.6, 6.6, 7.6, 8.6]:
    dens = stats.norm.pdf(yg, beta_hand[1] + beta_hand[0] * x0, sigma_hat_mle)
    ax.plot(x0 + 0.5 * dens / dens.max(), yg, color="tab:orange", lw=1.2)
ax.plot([], [], color="tab:orange", label=f"p(y | x) = N(line, σ̂²), σ̂ = {sigma_hat_mle:.2f}")
ax.set_xlabel("STAT1 (log2)"); ax.set_ylabel("MX1 (log2)"); ax.legend(fontsize=7, loc="upper left")
ax.set_title("the Gaussian regression model")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The orange curves are the model's conditional densities $p(y\mid x)$ at four values of
*STAT1*, drawn sideways: same width everywhere (σ̂ = 0.81 log2 units), centre on the line. The model treats every sample
as "line + Gaussian noise". Whether that is a good description is checked by residual diagnostics (Section 7).

### Maximum likelihood for $\boldsymbol\beta$ — every line

The samples are independent, so the likelihood is a product (Module 17), and the log turns it into a sum (Module 23):

$$\begin{aligned}
\ln p(\mathcal D\mid\boldsymbol\beta,\sigma^2) &= \ln\prod_{i=1}^n\frac{1}{\sigma\sqrt{2\pi}}\,e^{-\frac{1}{2\sigma^2}(y_i-\mathbf x_i^\top\boldsymbol\beta)^2}\\
&= \sum_{i=1}^n\ln\Big[\frac{1}{\sigma\sqrt{2\pi}}\,e^{-\frac{1}{2\sigma^2}(y_i-\mathbf x_i^\top\boldsymbol\beta)^2}\Big]\\
&= \sum_{i=1}^n\Big[-\ln\sigma - \tfrac12\ln(2\pi) - \frac{1}{2\sigma^2}(y_i-\mathbf x_i^\top\boldsymbol\beta)^2\Big]\\
&= -\frac n2\ln\sigma^2 - \frac n2\ln(2\pi) - \frac{1}{2\sigma^2}\sum_{i=1}^n(y_i-\mathbf x_i^\top\boldsymbol\beta)^2\\
&= -\frac n2\ln\sigma^2 - \frac n2\ln(2\pi) - \frac{1}{2\sigma^2}\|\mathbf y-X\boldsymbol\beta\|_2^2 .
\end{aligned}$$

*In words:* line 2 uses $\ln\prod = \sum\ln$; line 3 uses $\ln(ab) = \ln a+\ln b$, $\ln(1/a) = -\ln a$ and $\ln e^z = z$;
line 4 adds up the $n$ copies of the constants and uses $-\ln\sigma = -\tfrac12\ln\sigma^2$; line 5 recognises the sum of
squares as a squared norm.

Now differentiate with respect to $\boldsymbol\beta$. The first two terms do not contain $\boldsymbol\beta$; expand the
norm as in Section 3:

$$\begin{aligned}
\nabla_{\boldsymbol\beta}\ln p(\mathcal D\mid\boldsymbol\beta,\sigma^2) &= -\frac{1}{2\sigma^2}\,\nabla_{\boldsymbol\beta}\big(\mathbf y^\top\mathbf y - 2\boldsymbol\beta^\top X^\top\mathbf y + \boldsymbol\beta^\top X^\top X\boldsymbol\beta\big)\\
&= -\frac{1}{2\sigma^2}\big(-2X^\top\mathbf y + 2X^\top X\boldsymbol\beta\big) = \frac{1}{\sigma^2}\big(X^\top\mathbf y - X^\top X\boldsymbol\beta\big) \overset{!}{=} \mathbf 0\\
\Longrightarrow\quad \hat{\boldsymbol\beta}_{MLE} &= (X^\top X)^{-1}X^\top\mathbf y .
\end{aligned}$$

*In words:* the most likely line under Gaussian noise is exactly the least-squares line. Maximising
$-\frac{1}{2\sigma^2}\|\mathbf y-X\boldsymbol\beta\|^2$ is minimising $\|\mathbf y-X\boldsymbol\beta\|^2$, whatever σ is.
That is the probabilistic justification of "least **squares**": squares come from the $e^{-(\cdot)^2}$ of the Gaussian.

⚠️ **Common confusion — the constants.** The Gaussian density is $\frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{(y-\mu)^2}{2\sigma^2}}$:
the normalising factor has σ (not σ²) and the exponent has $2\sigma^2$ (not $2\sigma$). Writing $1/(\sigma^2\sqrt{2\pi})$
would give $-n\ln\sigma^2$ instead of $-\frac n2\ln\sigma^2$ and a wrong $\hat\sigma^2$. And the term $-\frac n2\ln(2\pi)$ may be
dropped only because it does not depend on the parameters. Write the sum of squares as $\|\mathbf y-X\boldsymbol\beta\|^2$
(or $(\mathbf y-X\boldsymbol\beta)^\top(\mathbf y-X\boldsymbol\beta)$), not "$(Y-X\beta)^2$", which is not defined for a vector.

### Maximum likelihood for $\sigma^2$

We do not know σ — and it is what tells us how uncertain any prediction is. Differentiate with respect to $\sigma^2$
(treat $s = \sigma^2$ as the variable; $\frac{d}{ds}\ln s = \frac1s$, $\frac{d}{ds}s^{-1} = -s^{-2}$):

$$\frac{\partial}{\partial\sigma^2}\Big[-\frac n2\ln\sigma^2-\frac{1}{2\sigma^2}\sum_{i=1}^n(y_i-\mathbf x_i^\top\boldsymbol\beta)^2\Big]
= -\frac{n}{2\sigma^2}+\frac12(\sigma^2)^{-2}\sum_{i=1}^n(y_i-\mathbf x_i^\top\boldsymbol\beta)^2 \overset{!}{=}0
\quad\Longrightarrow\quad \hat\sigma^2_{MLE} = \frac{1}{n}\sum_{i=1}^n\big(y_i-\mathbf x_i^\top\hat{\boldsymbol\beta}\big)^2 = \frac{\text{RSS}}{n}.$$

*In words:* multiply the equation by $2(\sigma^2)^2$ to get $-n\sigma^2 + \text{RSS} = 0$: the MLE of the noise variance
is the mean squared residual.

In [ ]:
# ── 5.2 Check: maximise the log-likelihood numerically and compare with the closed forms ──
def neg_log_lik(theta, X, y):
    """Negative Gaussian log-likelihood of a linear model; theta = (β_1, …, β_p, log σ)."""
    beta, log_sigma = theta[:-1], theta[-1]
    sigma2 = np.exp(2 * log_sigma)
    r_ = y - X @ beta
    return 0.5 * len(y) * np.log(2 * np.pi * sigma2) + (r_ @ r_) / (2 * sigma2)


fit = optimize.minimize(neg_log_lik, x0=np.array([0.0, 0.0, 0.0]), args=(X1, mx1), method="BFGS")
print("numerical MLE:   β_s = {:.4f}, β_0 = {:.4f}, σ² = {:.4f}".format(fit.x[0], fit.x[1], np.exp(2 * fit.x[2])))
print("closed forms:    β_s = {:.4f}, β_0 = {:.4f}, σ² = RSS/n = {:.4f}".format(beta_hand[0], beta_hand[1], e @ e / n))
print(f"maximum log-likelihood = {-fit.fun:.4f};  formula −(n/2)·ln(2π σ̂²) − n/2 = "
      f"{-n / 2 * np.log(2 * np.pi * e @ e / n) - n / 2:.4f}")

# the log-likelihood as a function of σ² alone, with β fixed at β̂ (a "profile")
s2_grid = np.linspace(0.2, 2.0, 300)
ll_profile = -n / 2 * np.log(2 * np.pi * s2_grid) - (e @ e) / (2 * s2_grid)
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(s2_grid, ll_profile, color="tab:purple")
ax.axvline(e @ e / n, color="k", ls="--", label=f"RSS/n = {e @ e / n:.3f}")
ax.axvline(e @ e / (n - 2), color="tab:orange", ls=":", label=f"RSS/(n−2) = {e @ e / (n - 2):.3f}")
ax.set_xlabel("σ²"); ax.set_ylabel("log-likelihood at β = β̂"); ax.legend()
ax.set_title("the likelihood is highest at σ² = RSS/n")
plt.tight_layout(); plt.show()

🔍 **What just happened?** A general-purpose optimiser, which knows nothing about normal equations, climbs the
log-likelihood to the same place: $\hat\beta_s = 0.9448$, $\hat\beta_0 = 0.9176$ and $\hat\sigma^2 = 0.6641$ = RSS/$n$ (so
$\hat\sigma = 0.81$). At the maximum the log-likelihood is $-\frac n2\ln(2\pi\hat\sigma^2)-\frac n2 = -29.14$, because there
$\text{RSS}/(2\hat\sigma^2) = n/2$. The plot shows the log-likelihood as a function of σ² with β held at $\hat{\boldsymbol\beta}$:
too small a σ² makes the observed residuals improbably large, too large a σ² spreads the probability too thin; the peak
is exactly at RSS/$n$. The orange line marks RSS/$(n-2) = 0.725$, slightly to the right.

**Is RSS/$n$ a good estimate?** Like the $\frac1n$ sample variance of [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb), it is **biased downwards**: the fitted line
is chosen to be close to *these* points, so the residuals are smaller than the true errors. Each estimated coefficient
uses up one degree of freedom, and the unbiased estimate divides by $n-p$ ($p$ = number of columns of $X$):

$$s^2 = \frac{\text{RSS}}{n-p}, \qquad E[s^2] = \sigma^2 .$$

*In words:* the same Bessel-type correction as $n-1$ for the sample variance — which is the special case $p = 1$ (a model
with only an intercept).

In [ ]:
# ── 5.3 The bias of RSS/n, by simulation (small n = 10, p = 2; SIMULATED) ──
rng_bias = np.random.default_rng(SEED + 53)
n_small, sigma_true = 10, 1.5
x_small = np.linspace(0, 9, n_small)
X_small = np.column_stack([x_small, np.ones(n_small)])
est_n, est_np = [], []
for _ in range(20_000):
    y_sim = 2.0 + 0.5 * x_small + rng_bias.normal(0, sigma_true, n_small)
    b_sim = np.linalg.lstsq(X_small, y_sim, rcond=None)[0]
    rss_sim = rss(b_sim, X_small, y_sim)
    est_n.append(rss_sim / n_small); est_np.append(rss_sim / (n_small - 2))
print(f"true σ² = {sigma_true ** 2:.3f}")
print(f"average of RSS/n       = {np.mean(est_n):.3f}   (theory: σ²·(n−p)/n = {sigma_true ** 2 * 8 / 10:.3f})")
print(f"average of RSS/(n − p) = {np.mean(est_np):.3f}")

🔍 **What just happened?** With $n = 10$ samples and $p = 2$ coefficients, RSS/$n$ averages **1.80** instead of the true
$\sigma^2 = 2.25$ — exactly the predicted factor $(n-p)/n = 0.8$ too small — while RSS/$(n-p)$ averages 2.25. The MLE is
consistent (the bias vanishes as $n\to\infty$) but in small biological experiments the correction matters, and all
standard regression software uses $s^2 = \text{RSS}/(n-p)$ for standard errors and tests.

### 📘 Lecture notes — the Gaussian model and maximum likelihood

- **Model:** $p(y_i\mid x_i) = \mathcal N(\beta_0+\beta_sx_i,\sigma^2)$ ⇔ $y_i = \beta_0+\beta_sx_i+\varepsilon_i$,
  $\varepsilon_i\sim\mathcal N(0,\sigma^2)$ i.i.d.
- **Log-likelihood:** $\ln p(\mathcal D\mid\boldsymbol\beta,\sigma^2) = \sum_i\ln\big[\frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{1}{2\sigma^2}(y_i-\mathbf x_i^\top\boldsymbol\beta)^2}\big]
  = -\frac n2\ln\sigma^2-\frac n2\ln2\pi-\frac{1}{2\sigma^2}\|\mathbf y-X\boldsymbol\beta\|^2$.
- **MLE of $\boldsymbol\beta$:** $\nabla_{\boldsymbol\beta} = -\frac{1}{2\sigma^2}(-2X^\top\mathbf y+2X^\top X\boldsymbol\beta) = \mathbf 0$ ⇒
  $\hat{\boldsymbol\beta}_{MLE} = (X^\top X)^{-1}X^\top\mathbf y$ — **least squares**.
- **MLE of $\sigma^2$:** $\frac{\partial}{\partial\sigma^2} = -\frac{n}{2\sigma^2}+\frac12(\sigma^2)^{-2}\text{RSS} = 0$ ⇒
  $\hat\sigma^2_{MLE} = \text{RSS}/n$ (biased; unbiased $s^2 = \text{RSS}/(n-p)$).
- **Why squares:** the Gaussian's $e^{-(\cdot)^2}$. Other noise models give other losses (Laplace noise → absolute
  errors; counts → Poisson/negative-binomial GLMs).

> 🎓 **Exam tip.** In the derivation, say *why* each term disappears: $-\frac n2\ln2\pi$ and $-\frac n2\ln\sigma^2$ do not depend on $\boldsymbol\beta$; $\frac{1}{2\sigma^2}>0$ does not move the arg max; the sign flip turns arg max into arg min.

### 🌉 Bridge — Change the noise, change the loss: generalised linear models

"Least squares = Gaussian MLE" is the first instance of a general recipe used throughout 02-620: **choose a distribution
for $y\mid\mathbf x$, write its log-likelihood, maximise**. A 0/1 outcome (disease or not) with a Bernoulli model gives
**logistic regression** ([BiologyToML Module 09](../../../../ml/tutorial/BiologyToML/notebooks/09_logistic_regression.ipynb)); read counts with a Poisson or
negative-binomial model give the count GLMs behind DESeq2 and edgeR
([BiologyToML Module 24](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)). The algebra of this section — products → sums of logs →
derivatives → normal-equation-like conditions — carries over, except that the equations are no longer linear and are
solved iteratively (Newton-type methods such as the Gauss–Newton iteration of [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb)).

---
## 6 · Inference and prediction

> 📍 **Where we are.** We have an estimate. How precise is it? Is the slope really different from zero? What value do we
> expect for a new sample? The answers combine expectation and covariance ([Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb)), the $t$ distribution ([Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb)) and
> confidence intervals ([Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb)).

**The estimate is a random vector.** $\hat{\boldsymbol\beta} = A\mathbf y$ with the fixed matrix $A = (X^\top X)^{-1}X^\top$.
Under the model, $\mathbf y = X\boldsymbol\beta+\boldsymbol\varepsilon$ with $E[\boldsymbol\varepsilon] = \mathbf 0$ and
$\operatorname{Cov}(\boldsymbol\varepsilon) = \sigma^2I$. Linearity of expectation and the rule
$\operatorname{Cov}(A\mathbf y) = A\operatorname{Cov}(\mathbf y)A^\top$ give

$$E[\hat{\boldsymbol\beta}] = AX\boldsymbol\beta = (X^\top X)^{-1}X^\top X\boldsymbol\beta = \boldsymbol\beta,\qquad
\operatorname{Cov}(\hat{\boldsymbol\beta}) = A\,\sigma^2I\,A^\top = \sigma^2(X^\top X)^{-1}X^\top X(X^\top X)^{-1} = \sigma^2(X^\top X)^{-1}.$$

*In words:* least squares is **unbiased**, and the variances of the coefficients are σ² times the diagonal of
$(X^\top X)^{-1}$ — small when the predictor values are spread out and numerous (large $X^\top X$).

**Standard errors, $t$-tests and confidence intervals.** Replacing σ² by $s^2 = \text{RSS}/(n-p)$:

$$\operatorname{SE}(\hat\beta_j) = s\sqrt{\big[(X^\top X)^{-1}\big]_{jj}},\qquad
T_j = \frac{\hat\beta_j - 0}{\operatorname{SE}(\hat\beta_j)}\sim t_{n-p}\ \text{ under } H_0:\beta_j = 0,\qquad
\hat\beta_j \pm t_{n-p,\,0.975}\,\operatorname{SE}(\hat\beta_j).$$

*In words:* exactly the one-sample $t$ machinery of [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb), with $n-p$ degrees of freedom because $p$ coefficients were
estimated. **$R^2$** $= 1-\text{RSS}/\text{TSS}$, with $\text{TSS} = \sum(y_i-\bar y)^2$, is the fraction of the variance of
$y$ explained by the model; for simple regression $R^2 = r^2$.

In [ ]:
# ── 6.1 Standard errors, t, p, confidence intervals and R² for MX1 ~ STAT1, checked with linregress ──
def ols_table(X, y, names):
    """Least-squares fit with the classical inference table (estimate, SE, t, p, 95% CI) and s², R²."""
    n_, p_ = X.shape
    XtX_inv_ = np.linalg.inv(X.T @ X)
    beta = XtX_inv_ @ X.T @ y
    resid = y - X @ beta
    s2 = resid @ resid / (n_ - p_)
    se_ = np.sqrt(s2 * np.diag(XtX_inv_))
    t_ = beta / se_
    tq = stats.t.ppf(0.975, n_ - p_)
    table_ = pd.DataFrame({"estimate": beta, "SE": se_, "t": t_, "p": 2 * stats.t.sf(np.abs(t_), n_ - p_),
                           "CI low": beta - tq * se_, "CI high": beta + tq * se_}, index=names)
    r2 = 1 - resid @ resid / np.sum((y - y.mean()) ** 2)
    return table_, s2, r2


tab1, s2_1, r2_1 = ols_table(X1, mx1, ["slope (STAT1)", "intercept"])
print(tab1.round(5).to_string())
print(f"s = √(RSS/(n−2)) = {np.sqrt(s2_1):.4f}, R² = {r2_1:.4f}, r² = {np.corrcoef(stat1, mx1)[0, 1] ** 2:.4f}")
print(f"linregress: slope SE {lr.stderr:.5f}, intercept SE {lr.intercept_stderr:.5f}, p {lr.pvalue:.3g}, r² {lr.rvalue ** 2:.4f}")

🔍 **What just happened?** The slope is **0.945 ± 0.167** (SE), so $t = 5.65$ on 22 degrees of freedom, $p = 1.1\times10^{-5}$,
and the 95 % confidence interval is **[0.60, 1.29]** log2 units of *MX1* per log2 unit of *STAT1*. The intercept has a
huge standard error (1.19) because $x = 0$ is far outside the data — the intercept is an extrapolation. $R^2 = 0.592 = r^2$:
*STAT1* "explains" 59 % of the variance of *MX1* across these samples. `linregress` reports the same SEs and $p$-value.

**Two identities worth knowing.** (1) Centring $\mathbf y$ and $\hat{\mathbf y}$ at $\bar y$ and applying Pythagoras
(Section 4) splits the total sum of squares: $\text{TSS} = \text{ESS} + \text{RSS}$ with
$\text{ESS} = \sum_i(\hat y_i-\bar y)^2$, so $R^2 = \text{ESS}/\text{TSS}$ is a ratio of squared lengths. (2) For one
predictor, the slope's $t$-statistic depends only on the correlation:

$$t = \frac{\hat\beta_s}{\operatorname{SE}(\hat\beta_s)} = \frac{r\sqrt{n-2}}{\sqrt{1-r^2}} ,$$

*In words:* testing "slope = 0" and testing "correlation = 0" are the same test — a strong correlation in a large sample
gives a large $t$.

In [ ]:
# ── 6.2 TSS = ESS + RSS, and t from the correlation alone ──
tss = np.sum((mx1 - mx1.mean()) ** 2)
ess = np.sum((y_hat - mx1.mean()) ** 2)
rss_1 = e @ e
print(f"TSS = {tss:.4f};  ESS + RSS = {ess:.4f} + {rss_1:.4f} = {ess + rss_1:.4f};  R² = ESS/TSS = {ess / tss:.4f}")
r_ = np.corrcoef(stat1, mx1)[0, 1]
print(f"t from the table = {tab1.loc['slope (STAT1)', 't']:.4f};  r·√(n−2)/√(1−r²) = {r_ * np.sqrt(n - 2) / np.sqrt(1 - r_ ** 2):.4f}")

🔍 **What just happened?** The total variation of *MX1* around its mean (TSS = 39.07) splits exactly into the part the line
reproduces (ESS = 23.13) and the part it leaves in the residuals (RSS = 15.94), giving $R^2 = 0.592$ again. And the
correlation $r = 0.769$ alone reproduces the slope's $t = 5.65$.

Do these formulas deliver what they promise? A confidence interval should cover the true slope 95 % of the time. Only a
simulation with a known truth can check that.

In [ ]:
# ── 6.3 Coverage of the 95% CI for the slope: 5,000 simulated experiments with a known truth ──
rng_cov = np.random.default_rng(SEED + 62)
true_b0, true_bs, true_sigma = 1.0, 0.8, 0.9          # SIMULATED regulator → target model
covered, t_stats = 0, []
for _ in range(5_000):
    y_sim = true_b0 + true_bs * stat1 + rng_cov.normal(0, true_sigma, n)   # same STAT1 design as the data
    tab_sim, _, _ = ols_table(X1, y_sim, ["slope", "intercept"])
    covered += tab_sim.loc["slope", "CI low"] <= true_bs <= tab_sim.loc["slope", "CI high"]
    t_stats.append((tab_sim.loc["slope", "estimate"] - true_bs) / tab_sim.loc["slope", "SE"])
print(f"95% CIs that contain the true slope {true_bs}: {covered / 5000:.4f}")
t_stats = np.array(t_stats)
print(f"(β̂_s − β_s)/SE: mean {t_stats.mean():.3f}, SD {t_stats.std():.3f} (t_22 SD = {np.sqrt(22 / 20):.3f}); "
      f"P(|T| > t_22,0.975) = {np.mean(np.abs(t_stats) > stats.t.ppf(0.975, 22)):.4f}")

🔍 **What just happened?** Over 5,000 simulated experiments the 95 % interval contained the true slope 0.8 about
**95 %** of the time, and the standardised estimates have the spread of a $t_{22}$ distribution (SD $\sqrt{22/20} = 1.049$).
The formulas are exact when the model holds — the noise is Gaussian with constant variance, and the samples independent.

### Predicting a new sample

For a new sample with predictor value $\tilde x$ (augmented row $\tilde{\mathbf x} = (\tilde x, 1)^\top$), the model gives the
**predictive distribution** $p(\tilde y\mid\tilde x) = \mathcal N(\tilde{\mathbf x}^\top\boldsymbol\beta,\ \sigma^2)$. Plugging
in the estimates gives a first answer, $\mathcal N(\tilde{\mathbf x}^\top\hat{\boldsymbol\beta}, \hat\sigma^2)$. But
$\hat{\boldsymbol\beta}$ is itself uncertain; since the new noise is independent of the data,

$$\operatorname{Var}\big(\tilde y - \tilde{\mathbf x}^\top\hat{\boldsymbol\beta}\big) = \sigma^2 + \tilde{\mathbf x}^\top\operatorname{Cov}(\hat{\boldsymbol\beta})\tilde{\mathbf x} = \sigma^2\big(1 + \tilde{\mathbf x}^\top(X^\top X)^{-1}\tilde{\mathbf x}\big).$$

*In words:* a new measurement is uncertain for two reasons — its own noise (the "1") and our uncertainty about the line
(the second term, which is smallest near the centre of the data and grows as we extrapolate). This gives a 95 %
**prediction interval** $\tilde{\mathbf x}^\top\hat{\boldsymbol\beta}\pm t_{n-p,0.975}\,s\sqrt{1+\tilde{\mathbf x}^\top(X^\top X)^{-1}\tilde{\mathbf x}}$, and,
without the "1", a **confidence band** for the mean line.

In [ ]:
# ── 6.4 Confidence band for the line vs prediction band for new samples ──
s1 = np.sqrt(s2_1)
XtX1_inv = np.linalg.inv(X1.T @ X1)
x_new = np.linspace(4.5, 10.0, 200)
Xn = np.column_stack([x_new, np.ones_like(x_new)])
lev = np.einsum("ij,jk,ik->i", Xn, XtX1_inv, Xn)            # x̃ᵀ(XᵀX)⁻¹x̃ for every x̃
mean_line = Xn @ beta_hand
tq = stats.t.ppf(0.975, n - 2)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.fill_between(x_new, mean_line - tq * s1 * np.sqrt(1 + lev), mean_line + tq * s1 * np.sqrt(1 + lev),
                color="tab:orange", alpha=0.2, label="95% prediction band (new sample)")
ax.fill_between(x_new, mean_line - tq * s1 * np.sqrt(lev), mean_line + tq * s1 * np.sqrt(lev),
                color="tab:blue", alpha=0.35, label="95% confidence band (mean line)")
ax.plot(x_new, mean_line, "k-", lw=1.5)
ax.scatter(stat1, mx1, s=16, color="k", zorder=3, label="data")
ax.set_xlabel("STAT1 (log2)"); ax.set_ylabel("MX1 (log2)"); ax.legend(fontsize=7, loc="upper left")
ax.set_title("uncertainty about the line vs about a new sample")
plt.tight_layout(); plt.show()
lev_d = np.einsum("ij,jk,ik->i", X1, XtX1_inv, X1)
inside = np.abs(mx1 - X1 @ beta_hand) <= tq * s1 * np.sqrt(1 + lev_d)
print(f"data points inside the 95% prediction band: {inside.sum()} of {n}")
x_t = np.array([8.0, 1.0])
half = tq * s1 * np.sqrt(1 + x_t @ XtX1_inv @ x_t)
print(f"new sample with STAT1 = 8: predicted MX1 = {x_t @ beta_hand:.2f}, 95% prediction interval "
      f"[{x_t @ beta_hand - half:.2f}, {x_t @ beta_hand + half:.2f}]")

🔍 **What just happened?** The blue band (where the *mean* MX1 level plausibly lies) is narrow near the centre of the
data and flares out at the ends; the orange band (where a *new sample* plausibly lies) is much wider because it adds the
sample's own noise. All 24 samples fall inside the 95 % prediction band (on average about 23 of 24 would). For a new
sample with *STAT1* = 8, the model predicts *MX1* = 8.48 with a 95 % prediction interval of [6.65, 10.31] — wide, because the data
scatter a lot around the line.

### 📘 Lecture notes — inference and prediction

- $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf y$ is **unbiased**, with $\operatorname{Cov}(\hat{\boldsymbol\beta}) = \sigma^2(X^\top X)^{-1}$.
- $\operatorname{SE}(\hat\beta_j) = s\sqrt{[(X^\top X)^{-1}]_{jj}}$, $s^2 = \text{RSS}/(n-p)$; $t = \hat\beta_j/\operatorname{SE}\sim t_{n-p}$
  under $H_0:\beta_j = 0$; 95 % CI $\hat\beta_j\pm t_{n-p,0.975}\operatorname{SE}$. **$R^2 = 1-\text{RSS}/\text{TSS}$** ($=r^2$ for one predictor).
- *MX1* ~ *STAT1*: $\hat\beta_s = 0.945$, SE 0.167, $t = 5.65$, $p = 1.1\times10^{-5}$, CI [0.60, 1.29], $R^2 = 0.59$.
- **Prediction** for a new $\tilde x$: $p(\tilde y\mid\tilde x) = \mathcal N(\tilde{\mathbf x}^\top\boldsymbol\beta,\sigma^2)$;
  with estimated $\boldsymbol\beta$ the variance is $\sigma^2(1+\tilde{\mathbf x}^\top(X^\top X)^{-1}\tilde{\mathbf x})$ →
  prediction intervals are wider than confidence bands and widen away from the data.

> 🎓 **Exam tip.** Distinguish the three intervals: CI for a coefficient, confidence band for the mean response, prediction interval for a new observation (the widest).

✏️ **Your turn.** Compute the 95 % prediction interval at *STAT1* = 12, far outside the data. Why should you not trust it
even more than its width suggests?

---
## 7 · Multiple regression: the insulin study

> 📍 **Where we are.** With several predictors, nothing in the formulas changes — $X$ just has more columns. What changes
> is the *interpretation*: each coefficient is an effect **holding the other predictors fixed**.

The model for the $N = 120$ adults:

$$\text{insulin}_i = \beta_0 + \beta_1\,\text{BMI}_i + \beta_2\,\text{age}_i + \beta_3\,\text{glucose}_i + \varepsilon_i,\qquad \varepsilon_i\sim\mathcal N(0,\sigma^2),$$

*In words:* fasting insulin is a baseline plus a contribution per kg/m² of BMI, per year of age and per mg/dL of glucose,
plus noise; $X$ is $120\times4$ (a column of ones and three clinical columns). We fit it three ways: the normal equations,
`np.linalg.lstsq` (which uses the SVD, [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb)), and **gradient descent** ([Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb)) on the RSS.

**Why standardise for gradient descent?** The columns have very different scales (age ≈ 50, glucose ≈ 90, the column of
ones = 1), so the RSS bowl is extremely elongated (condition number ≈ 10³ for $X$, ≈ 10⁶ for $X^\top X$) and plain
gradient descent zig-zags for thousands of steps. Rescaling each predictor to mean 0 and SD 1 makes the bowl almost round;
afterwards we convert the coefficients back to the original units.

In [ ]:
# ── 7.1 Three solvers for the insulin model (SIMULATED data with known truth) ──
names_ins = ["intercept", "bmi", "age", "glucose"]
beta_ne = np.linalg.solve(X_ins.T @ X_ins, X_ins.T @ y_ins)          # normal equations
beta_ls = np.linalg.lstsq(X_ins, y_ins, rcond=None)[0]                 # SVD-based least squares

# gradient descent on standardised predictors (Module 15): minimise (1/N)‖yc − Zγ‖²
feats = X_ins[:, 1:]
mu_f, sd_f = feats.mean(axis=0), feats.std(axis=0)
Z = (feats - mu_f) / sd_f
yc = y_ins - y_ins.mean()
gamma, lr_gd, history = np.zeros(3), 0.1, []
for step in range(300):
    grad = -2 / len(yc) * Z.T @ (yc - Z @ gamma)
    gamma -= lr_gd * grad
    history.append(np.mean((yc - Z @ gamma) ** 2))
beta_gd_slopes = gamma / sd_f                                           # back to original units
beta_gd = np.concatenate([[y_ins.mean() - mu_f @ beta_gd_slopes], beta_gd_slopes])

comp = pd.DataFrame({"normal equations": beta_ne, "lstsq (SVD)": beta_ls, "gradient descent (300 steps)": beta_gd,
                     "TRUE (simulation)": [INSULIN_TRUE_COEFFICIENTS[k] for k in names_ins]}, index=names_ins)
print(comp.round(4).to_string())
print(f"condition number of XᵀX: original {np.linalg.cond(X_ins.T @ X_ins):.2e}, standardised "
      f"{np.linalg.cond(Z.T @ Z):.2f}")
print(f"mean squared error after 1, 10, 100, 300 GD steps: "
      f"{history[0]:.3f}, {history[9]:.3f}, {history[99]:.4f}, {history[299]:.4f}")

🔍 **What just happened?** The three solvers agree to four decimals (gradient descent after 300 steps on the standardised
problem, whose $Z^\top Z$ has condition number about 3 instead of about $10^6$). The estimates are close to, but not equal to,
the truth: BMI 0.563 (true 0.55), age 0.034 (0.04), glucose 0.051 (0.06), intercept −2.94 (−4). That is sampling
noise — 120 patients do not pin the coefficients down exactly. How exactly? The inference table tells us.

In [ ]:
# ── 7.2 The coefficient table: SE, t, p, 95% CI — and does each CI contain the truth? ──
tab_ins, s2_ins, r2_ins = ols_table(X_ins, y_ins, names_ins)
tab_ins["truth"] = [INSULIN_TRUE_COEFFICIENTS[k] for k in names_ins]
tab_ins["truth in CI?"] = (tab_ins["CI low"] <= tab_ins["truth"]) & (tab_ins["truth"] <= tab_ins["CI high"])
print(tab_ins.round(4).to_string())
print(f"s = {np.sqrt(s2_ins):.3f} µU/mL (true σ = {INSULIN_TRUE_SIGMA}), R² = {r2_ins:.3f}, df = {len(y_ins) - 4}")

🔍 **What just happened?** Every 95 % interval contains the true coefficient. BMI has a strong, precisely estimated effect
($\hat\beta = 0.563$, SE 0.062, $t = 9.1$, $p<10^{-14}$): each extra kg/m² goes with 0.56 µU/mL more insulin at the same age
and glucose. Age and glucose have **real** effects in the simulation, yet their $p$-values are 0.057 and 0.074 — neither is
"significant" at 0.05. That is the power problem of [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb) in action: with 120 patients and $\sigma = 2.5$, effects of this
size are often missed. "Not significant" never meant "no effect". The residual SD $s = 2.69$ estimates the true 2.5, and the
three clinical variables explain $R^2 = 0.52$ of the variance of insulin.

**Are the model's assumptions plausible?** Residual diagnostics check them: residuals vs fitted values should show no
pattern (linearity, constant variance); a normal QQ-plot of the residuals should be a straight line (Gaussian noise).

In [ ]:
# ── 7.3 Residual diagnostics ──
fitted_ins = X_ins @ beta_ne
res_ins = y_ins - fitted_ins
std_res = np.sort(res_ins / np.sqrt(s2_ins))
theo_q = stats.norm.ppf((np.arange(1, len(std_res) + 1) - 0.5) / len(std_res))   # normal quantiles (Module 18)
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.3))
axes[0].scatter(fitted_ins, res_ins, s=12, alpha=0.7)
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_xlabel("fitted insulin (µU/mL)"); axes[0].set_ylabel("residual"); axes[0].set_title("residuals vs fitted")
axes[1].scatter(theo_q, std_res, s=12, alpha=0.7)
axes[1].plot([-3, 3], [-3, 3], "r--", lw=1)
axes[1].set_xlabel("standard normal quantile"); axes[1].set_ylabel("sorted standardised residual")
axes[1].set_title("normal QQ-plot")
axes[2].hist(res_ins, bins=18, density=True, color="0.7", edgecolor="white")
g_ = np.linspace(-8, 8, 200)
axes[2].plot(g_, stats.norm.pdf(g_, 0, np.sqrt(s2_ins)), "r-", label="N(0, s²)")
axes[2].set_xlabel("residual (µU/mL)"); axes[2].set_title("residual histogram"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"mean residual = {res_ins.mean():.2e} (zero by construction: Xᵀe = 0);  "
      f"correlation(residual, fitted) = {np.corrcoef(res_ins, fitted_ins)[0, 1]:.2e}")

🔍 **What just happened?** The residuals scatter evenly around zero with no curve or funnel; the QQ-plot points lie close
to the diagonal; and the histogram matches the normal curve. The residuals average exactly zero and are exactly
uncorrelated with the fitted values — not a finding but a consequence of $X^\top\mathbf e = \mathbf 0$ (Section 4). Here the
assumptions hold because the data were simulated from this very model; with real data, curved residual plots or funnel
shapes are common and call for transformations (e.g. log insulin) or other models.

**Omitted variables.** What if the study had not measured glucose? Glucose rises with BMI in this population, so part of
glucose's effect would be attributed to BMI.

In [ ]:
# ── 7.4 Omitted-variable bias: drop glucose ──
tab_noglu, _, r2_noglu = ols_table(X_ins[:, :3], y_ins, names_ins[:3])
print(tab_noglu[["estimate", "SE", "p"]].round(4).to_string())
print(f"BMI coefficient: with glucose {tab_ins.loc['bmi', 'estimate']:.3f}, without {tab_noglu.loc['bmi', 'estimate']:.3f}; "
      f"corr(BMI, glucose) = {np.corrcoef(insulin['bmi'], insulin['glucose'])[0, 1]:.3f}")
slope_glu_on_bmi = np.polyfit(insulin["bmi"], insulin["glucose"], 1)[0]
print(f"glucose rises {slope_glu_on_bmi:.3f} mg/dL per kg/m² of BMI; "
      f"{tab_ins.loc['glucose', 'estimate']:.4f} × {slope_glu_on_bmi:.3f} = {tab_ins.loc['glucose', 'estimate'] * slope_glu_on_bmi:.3f} "
      f"≈ the increase of the BMI coefficient")

# the same phenomenon in the expression data: MX1 ~ STAT1, with and without the infection status
X_conf = np.column_stack([stat1, infected.astype(float), np.ones(n)])
tab_conf, _, _ = ols_table(X_conf, mx1, ["slope (STAT1)", "infected (0/1)", "intercept"])
print("\nMX1 ~ STAT1 + infected:\n", tab_conf[["estimate", "SE", "p"]].round(4).to_string())

🔍 **What just happened?** Without glucose in the model, the BMI coefficient rises from 0.563 to **0.612**. The increase is
close to (glucose effect) × (how much glucose rises per unit BMI): $0.051\times0.87\approx0.044$ vs the observed 0.049
(the age coefficient shifts too, from 0.034 to 0.046, because glucose also rises with age). A coefficient is always "the
effect of this variable **given the others in the model**"; change the others and it changes.

The expression data show the same effect much more dramatically. Adding the infection status (0/1) to the regression of
*MX1* on *STAT1* turns the slope from **+0.94** into **−0.47** (SE 0.20): almost all of the association came from infection
raising both genes — a common cause (confounder), not regulation. (The simulator indeed has no STAT1 → MX1 mechanism;
within a condition, the remaining association reflects tissue differences.) This is why regression results from
observational data are **associations**, and why biologists confirm regulatory links by perturbation experiments
(knock-downs, CRISPR), which control $x$ for real — see
[BiologyToML Module 41 · Perturbation and causal inference](../../../../ml/tutorial/BiologyToML/notebooks/41_perturbation_causal_inference.ipynb).

### 📘 Lecture notes — multiple regression

- **Model:** $\mathbf y = X\boldsymbol\beta+\boldsymbol\varepsilon$ with $X = [\mathbf 1, \mathbf x_1,\dots,\mathbf x_J]$
  ($n\times(J+1)$); same formulas: $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf y$, SEs from $s^2(X^\top X)^{-1}$,
  $t_{n-p}$ tests, $p = J+1$.
- **Interpretation:** $\beta_j$ = average change of $y$ per unit of $x_j$ **holding the other predictors fixed**.
- **Insulin study** ($N=120$): BMI 0.563 (SE 0.062), age 0.034 (SE 0.018), glucose 0.051 (SE 0.028), $s = 2.69$, $R^2 = 0.52$;
  all CIs contain the truth; true but small effects can be non-significant (power).
- **Solvers:** normal equations (`solve`), `lstsq` (SVD; also handles rank deficiency), gradient descent (standardise first).
- **Diagnostics:** residuals vs fitted (no pattern), QQ-plot (straight line).
- **Omitted variables/confounders** change coefficients: BMI 0.563 → 0.612 without glucose; *STAT1* slope +0.94 → −0.47
  after adjusting for infection.

> 🎓 **Exam tip.** When asked to "interpret $\hat\beta_j$", always include the units and the phrase "holding the other variables in the model constant" — and never call an observational coefficient a causal effect without saying why.

---
## 8 · Two special cases: a 0/1 predictor, and a genome-wide scan

> 📍 **Where we are.** Regression contains the tests of [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb) and [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb) as special cases. Seeing them inside one model is
> the best way to remember them.

**A 0/1 predictor is a two-group comparison.** Let $x_i = 1$ for infected and 0 for control samples (an *indicator*
variable). Then the model says: control mean $=\beta_0$, infected mean $=\beta_0+\beta_s$. With $n_1$ infected and $n_0$
control samples, the normal equations are

$$\begin{bmatrix} n_1 & n_1\\ n_1 & n\end{bmatrix}\begin{bmatrix}\beta_s\\ \beta_0\end{bmatrix} = \begin{bmatrix}\sum_{\text{inf}}y_i\\ \sum_{\text{all}}y_i\end{bmatrix}
\quad\Longrightarrow\quad \hat\beta_0 = \bar y_{\text{control}},\qquad \hat\beta_s = \bar y_{\text{infected}} - \bar y_{\text{control}} .$$

*In words:* (subtract the first equation from the second: $n_0\beta_0 = \sum_{\text{control}}y_i$.) The intercept is the
control mean and the slope is the difference of the group means — and its $t$-statistic is **Student's two-sample
$t$-test** with pooled variance and $n-2$ degrees of freedom.

In [ ]:
# ── 8.1 Regression on an infection indicator = the two-sample t-test (MX1 in spleen, SIMULATED) ──
sp = (samples["tissue"] == "spleen").to_numpy()
x01 = infected[sp].astype(float)
y_sp = mx1[sp]
tab01, _, _ = ols_table(np.column_stack([x01, np.ones(sp.sum())]), y_sp, ["infected (slope)", "intercept"])
print(tab01[["estimate", "SE", "t", "p"]].round(6).to_string())
print(f"control mean = {y_sp[x01 == 0].mean():.4f};  infected − control = {y_sp[x01 == 1].mean() - y_sp[x01 == 0].mean():.4f}")
student = stats.ttest_ind(y_sp[x01 == 1], y_sp[x01 == 0], equal_var=True)
welch = stats.ttest_ind(y_sp[x01 == 1], y_sp[x01 == 0], equal_var=False)
print(f"Student's t-test: t = {student.statistic:.4f}, p = {student.pvalue:.6f}   (df = 6)")
print(f"Welch's t-test:   t = {welch.statistic:.4f}, p = {welch.pvalue:.6f}   (df ≈ {welch.df:.2f})")

🔍 **What just happened?** The regression intercept is the control mean (6.18 log2 units) and the slope is the
infected − control difference (**2.755**); its $t = 8.581$ and $p = 1.38\times10^{-4}$ are **identical** to Student's
two-sample $t$-test. Welch's test (Module 25) gives the same $t$ here — with equal group sizes the two standard-error
formulas coincide — but a slightly larger $p$ (0.00026) because it uses fewer degrees of freedom (5.34 instead of 6). With
several groups (three tissues) the same idea, with one indicator column per extra group, gives the analysis of variance
(ANOVA). "One indicator per *extra* group" matters — here is why.

In [ ]:
# ── 8.2 A three-level factor: tissue → baseline IL6 expression, and the dummy-variable trap (SIMULATED controls) ──
ctrl = ~infected
il6 = X_expr[ctrl, genes.index("IL6")]
tis = samples.loc[ctrl, "tissue"].to_numpy()
ind = {t: (tis == t).astype(float) for t in ["spleen", "kidney", "liver"]}
X_trap = np.column_stack([np.ones(ctrl.sum()), ind["spleen"], ind["kidney"], ind["liver"]])
print(f"intercept + 3 indicators: {X_trap.shape[1]} columns, rank {np.linalg.matrix_rank(X_trap)};  "
      f"X·(1, −1, −1, −1) = {X_trap @ np.array([1, -1, -1, -1.0])}")
X_ref = np.column_stack([np.ones(ctrl.sum()), ind["kidney"], ind["liver"]])        # spleen = reference level
b_ref = np.linalg.solve(X_ref.T @ X_ref, X_ref.T @ il6)
print("reference coding (spleen as baseline):",
      {k: round(float(v), 3) for k, v in zip(["intercept", "kidney − spleen", "liver − spleen"], b_ref)})
print("group means:", {t: round(float(il6[tis == t].mean()), 3) for t in ["spleen", "kidney", "liver"]})

🔍 **What just happened?** With an intercept **and** an indicator for every tissue, the columns are dependent — the three
indicators add up to the column of ones — so $X$ has 4 columns but rank 3, and $(1,-1,-1,-1)$ is in its null space: the
**dummy-variable trap**, the same rank problem as the genotype-coding trap of [Module 11 · 🧱 Linear independence, span, basis and rank](11_independence_basis_rank.ipynb). Dropping one indicator (here spleen,
the *reference level*) restores full rank, and the coefficients become interpretable: the intercept 4.397 is the spleen
mean of baseline *IL6* expression, and the other two are the kidney and liver means minus the spleen mean (+2.721 and
+3.947). Comparing several group means with one model like this is one-way ANOVA.

### 🌉 Bridge — A genome-wide association scan is one regression per SNP

In a GWAS of a quantitative trait, each SNP's genotype (0/1/2 copies of an allele) is the predictor of a simple regression;
the slope is the **effect per allele**, tested with the $t$-test of Section 6, and the thousands of $p$-values are corrected
for multiple testing ([Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb)). A **polygenic score** adds up a person's genotypes weighted by the estimated effects.
Here is a miniature version: 200 SNPs, 600 people, 3 causal SNPs.

In [ ]:
# ── 8.3 A miniature GWAS: 200 SNPs × 600 people, 3 causal SNPs (SIMULATED) ──
rng_gw = np.random.default_rng(SEED + 82)
n_people, n_snps = 600, 200
freqs = rng_gw.uniform(0.1, 0.5, n_snps)
G = np.column_stack([simulate_genotypes(n_people, f, seed=1000 + j) for j, f in enumerate(freqs)])
causal = {17: 0.45, 88: -0.35, 150: 0.30}                         # SNP index: true effect per allele
trait = sum(b * G[:, j] for j, b in causal.items()) + rng_gw.normal(0, 1, n_people)

pvals, betas = np.empty(n_snps), np.empty(n_snps)
for j in range(n_snps):
    Xj = np.column_stack([G[:, j], np.ones(n_people)])
    tab_j, _, _ = ols_table(Xj, trait, ["snp", "intercept"])
    betas[j], pvals[j] = tab_j.loc["snp", "estimate"], tab_j.loc["snp", "p"]

order = np.argsort(pvals)
m_t, q = n_snps, 0.05
passed = pvals[order] <= np.arange(1, m_t + 1) / m_t * q          # Benjamini–Hochberg (Module 26)
hits = order[:np.max(np.nonzero(passed)[0]) + 1] if passed.any() else np.array([], dtype=int)
print("top 5 SNPs by p-value:", [(int(j), f"β̂ = {betas[j]:+.3f}", f"p = {pvals[j]:.1e}") for j in order[:5]])
print("BH discoveries at q = 0.05:", sorted(hits.tolist()), "  true causal SNPs:", sorted(causal))
print("Bonferroni discoveries (p ≤ 0.05/200):", sorted(np.nonzero(pvals <= 0.05 / m_t)[0].tolist()))
prs = G[:, hits] @ betas[hits]                                     # polygenic score from the discovered SNPs
print(f"correlation of the polygenic score with the trait: r = {np.corrcoef(prs, trait)[0, 1]:.3f}")

🔍 **What just happened?** One simple regression per SNP — the same `ols_table` as for *MX1* — produced 200 slopes and
$p$-values. The three causal SNPs (17, 88, 150) have the smallest $p$-values by many orders of magnitude, with estimated
per-allele effects close to the true +0.45, −0.35 and +0.30; Benjamini–Hochberg and Bonferroni both select exactly these
three. A polygenic score built from them correlates with the trait at about $r\approx0.3$ — real but modest predictive
power, as for most complex human traits. Real GWAS have millions of SNPs, correlated neighbours (linkage disequilibrium)
and population structure to adjust for: see [BiologyToML Module 06 · Linear regression and GWAS](../../../../ml/tutorial/BiologyToML/notebooks/06_linear_regression_gwas.ipynb)
and [Module 35 · Statistical genetics](../../../../ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb).

✏️ **Your turn.** Halve the effect sizes in `causal` (or reduce `n_people` to 200). How many causal SNPs survive BH, and
how many false positives appear?

---
## 9 · MAP estimation = ridge regression

> 📍 **Where we are.** [Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb) added a prior to the likelihood. Do the same for regression and you get one of the most
> important tools of high-dimensional biology.

**Prior.** Assume the coefficients are a priori small: $\boldsymbol\beta\sim\mathcal N(\mathbf 0,\tau^2I)$, i.e. independent
$\beta_j\sim\mathcal N(0,\tau^2)$ (for centred data we leave the intercept out of the prior). By Bayes' rule the log
posterior is log-likelihood + log prior + constant:

$$\begin{aligned}
\ln p(\boldsymbol\beta\mid\mathcal D) &= -\frac{1}{2\sigma^2}\|\mathbf y-X\boldsymbol\beta\|^2 - \frac{1}{2\tau^2}\|\boldsymbol\beta\|^2 + \text{const}\\
\hat{\boldsymbol\beta}_{MAP} &= \underset{\boldsymbol\beta}{\arg\min}\ \|\mathbf y-X\boldsymbol\beta\|^2 + \lambda\|\boldsymbol\beta\|^2,\qquad \lambda = \frac{\sigma^2}{\tau^2}.
\end{aligned}$$

*In words:* multiply by $-2\sigma^2$ (a negative constant turns arg max into arg min): the MAP estimate minimises the squared
error **plus a penalty** on the size of the coefficients. A tight prior (small τ) means a large λ and strong shrinkage.

**Solution.** The gradient is $-2X^\top\mathbf y + 2X^\top X\boldsymbol\beta + 2\lambda\boldsymbol\beta$ (using
$\nabla\|\boldsymbol\beta\|^2 = 2\boldsymbol\beta$); setting it to zero,

$$(X^\top X+\lambda I)\,\boldsymbol\beta = X^\top\mathbf y\qquad\Longrightarrow\qquad \hat{\boldsymbol\beta}_{\text{ridge}} = (X^\top X+\lambda I)^{-1}X^\top\mathbf y .$$

*In words:* ridge regression adds λ to the diagonal of $X^\top X$. **This matrix is always invertible for λ > 0:** the
eigenvalues of $X^\top X$ are ≥ 0 (positive semidefinite, [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)) and adding $\lambda I$ shifts every eigenvalue by λ, so
all become ≥ λ > 0. Even with more genes than samples — when $X^\top X$ is certainly singular — ridge has a unique answer.

In [ ]:
# ── 9.1 More genes than samples: predict MX1 from 40 other genes in 24 samples (SIMULATED) ──
rng_r = np.random.default_rng(SEED + 91)
others = [j for j in range(len(genes)) if genes[j] not in ("MX1",)]
pick = rng_r.choice(others, size=40, replace=False)
Xg = X_expr[:, pick]
Xg_c = (Xg - Xg.mean(axis=0)) / Xg.std(axis=0)                     # centred, standardised genes
y_c = mx1 - mx1.mean()                                             # centred target: no intercept needed
G_ = Xg_c.T @ Xg_c
eig_G = np.linalg.eigvalsh(G_)
print(f"X is {Xg_c.shape[0]} × {Xg_c.shape[1]}: rank {np.linalg.matrix_rank(Xg_c)}; "
      f"{np.sum(eig_G < 1e-8)} of the 40 eigenvalues of XᵀX are 0 → XᵀX is singular")
for lam in [0.1, 1.0, 10.0]:
    eig_shift = np.linalg.eigvalsh(G_ + lam * np.eye(40))
    print(f"λ = {lam:5.1f}: smallest eigenvalue of XᵀX + λI = {eig_shift.min():.4f}")


def ridge(X, y, lam):
    """Ridge / MAP estimate (XᵀX + λI)⁻¹Xᵀy for centred X and y."""
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)


lams = np.logspace(-2, 3, 60)
path = np.array([ridge(Xg_c, y_c, lam) for lam in lams])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(lams, path, lw=1)
axes[0].set_xscale("log"); axes[0].set_xlabel("λ = σ²/τ²"); axes[0].set_ylabel("coefficient")
axes[0].set_title("ridge path: coefficients shrink towards 0 as λ grows")
# leave-one-out prediction error for each λ
loo = []
for lam in lams:
    errs = []
    for i in range(len(y_c)):
        keep = np.arange(len(y_c)) != i
        b_ = ridge(Xg_c[keep], y_c[keep] - y_c[keep].mean(), lam)
        errs.append((y_c[i] - y_c[keep].mean() - Xg_c[i] @ b_) ** 2)
    loo.append(np.mean(errs))
best = lams[int(np.argmin(loo))]
axes[1].plot(lams, loo, "k-"); axes[1].axvline(best, color="tab:red", ls="--", label=f"best λ ≈ {best:.2f}")
loo_mean = np.mean([(y_c[i] - np.delete(y_c, i).mean()) ** 2 for i in range(len(y_c))])   # mean-only model
axes[1].axhline(loo_mean, color="0.5", ls=":", label="predicting the mean only")
axes[1].set_xscale("log"); axes[1].set_xlabel("λ"); axes[1].set_ylabel("leave-one-out squared error")
axes[1].set_title("choosing λ by leave-one-out cross-validation"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"best λ ≈ {best:.2f}: LOO error {min(loo):.3f} vs {loo_mean:.3f} for the mean-only prediction; "
      f"at λ = 0.01: {loo[0]:.3f}")

🔍 **What just happened?** With 24 samples and 40 genes, $X$ has rank 23 (centring uses up one dimension), so 17 of the 40
eigenvalues of $X^\top X$ are zero and ordinary least squares has infinitely many perfect-fit solutions. Adding λ = 0.1,
1 or 10 lifts the smallest eigenvalue to exactly λ, as the eigenvalue argument predicts, and the ridge solution is unique.
The **ridge path** (left) shows all 40 coefficients shrinking smoothly towards 0 as the prior tightens. The right panel
chooses λ by **leave-one-out cross-validation**: fit on 23 samples, predict the 24th, repeat. Very small λ overfits (error 1.72
at λ = 0.01, hardly better than predicting the mean, 1.77), very large λ shrinks everything to zero (the error climbs back
towards that of the mean); the best λ (≈ 7.6)
roughly halves the error of predicting the mean (0.82 vs 1.77) — the infection-response genes among the 40 carry the
information.

Ridge also rescues the perfectly collinear insulin design of Section 4.3 (glucose entered twice, in mg/dL and mmol/L),
where least squares had infinitely many solutions.

In [ ]:
# ── 9.2 Ridge on the collinear insulin design: a unique answer that shares the glucose effect ──
F_dup = X_dup[:, 1:]                                                # bmi, age, glucose (mg/dL), glucose (mmol/L)
F_c = (F_dup - F_dup.mean(axis=0)) / F_dup.std(axis=0)             # standardise (the two glucose columns become identical)
y_ins_c = y_ins - y_ins.mean()
for lam in [1e-6, 1.0, 10.0]:
    g_ = ridge(F_c, y_ins_c, lam)
    print(f"λ = {lam:g}: standardised coefficients (bmi, age, glucose mg/dL, glucose mmol/L) = {np.round(g_, 4)}")
b_std = np.linalg.lstsq(np.column_stack([F_c[:, :3]]), y_ins_c, rcond=None)[0]
print(f"least squares with glucose once (standardised): {np.round(b_std, 4)}")

🔍 **What just happened?** After standardising, the two glucose columns are *identical*, so least squares cannot tell them
apart. Ridge chooses, among all equally good fits, the one with the smallest $\|\boldsymbol\beta\|^2$ — which splits the
glucose effect **equally** between the two copies (each gets half of the single-glucose coefficient, 0.51 → 0.26 + 0.26 at
tiny λ). Larger λ shrinks the large BMI coefficient most (2.51 → 2.27 at λ = 10) and moves some weight onto the glucose
columns, which are correlated with BMI (0.26 → 0.30 each): ridge prefers many moderate coefficients to a few large ones.
This is how ridge behaves with groups of correlated genes in an expression matrix: it spreads the weight across them
instead of picking one arbitrarily (the lasso, by contrast, tends to pick one).

### 📘 Lecture notes — two special cases and ridge regression

- **0/1 predictor:** $\hat\beta_0 = \bar y_{\text{control}}$, $\hat\beta_s = \bar y_{\text{infected}}-\bar y_{\text{control}}$; the
  slope's $t$-test **is** Student's two-sample $t$-test ($n-2$ df). Example: *MX1* in spleen, $\hat\beta_s = 2.755$, $t = 8.58$.
- **GWAS:** one regression per SNP (slope = effect per allele), then multiple-testing correction; polygenic score
  $=\sum_j\hat\beta_jg_j$.
- **MAP with prior $\boldsymbol\beta\sim\mathcal N(\mathbf 0,\tau^2I)$** = ridge: minimise $\|\mathbf y-X\boldsymbol\beta\|^2+\lambda\|\boldsymbol\beta\|^2$,
  $\lambda = \sigma^2/\tau^2$; $\hat{\boldsymbol\beta} = (X^\top X+\lambda I)^{-1}X^\top\mathbf y$; eigenvalues shift by λ ⇒
  always invertible, even when $J > n$. Choose λ by cross-validation.

> 🎓 **Exam tip.** Derive ridge in two lines from the MAP objective, and explain with eigenvalues why $X^\top X+\lambda I$ is invertible.

### 🌉 Bridge — Shrinkage everywhere

Ridge regression (Hoerl & Kennard 1970) is the Gaussian-prior member of a family: a Laplace prior gives the **lasso**,
which sets many coefficients exactly to zero and is used to select a few predictive genes or SNPs
([BiologyToML Module 07 · Ridge, lasso and sparse regression](../../../../ml/tutorial/BiologyToML/notebooks/07_ridge_lasso_sparse_regression.ipynb)). Choosing λ by
cross-validation is the subject of [Module 23 · Model selection](../../../../ml/tutorial/BiologyToML/notebooks/23_model_selection.ipynb). Shrinkage of noisy
per-gene estimates towards a common value is also what DESeq2 does with fold changes — the empirical-Bayes idea of [Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb).

---
## 10 · The whole course in one model

> 📍 **Where we are.** Every part of 02-680 has now appeared in linear regression. This section collects them in one
> table, and one code cell checks a dozen of them on the insulin fit.

| Part of the course | Concept | Where it appears in linear regression | Module |
|---|---|---|---|
| Logic | propositions, implication, quantifiers | the decision rule "reject $H_0$ ⇔ $p\le\alpha$"; "if the model holds, then the CI covers β with probability 0.95"; "$X^\top X$ invertible ⇔ columns independent" | [Module 02 · 🔣 Logic: the grammar of precise reasoning](02_logic.ipynb) |
| Sets and functions | sets, subsets, functions $f:S\to T$ | the data set $\{(x_i,y_i)\}$; the model as a function $\mathbb R\to\mathbb R$; sets of discovered vs true genes | [Module 03 · 🧺 Sets and functions: collections and mappings](03_sets_and_functions.ipynb) |
| Tuples and sequences | ordered pairs, $\mathbb R^n$ | each sample is a tuple $(x_i, y_i)$; a GWAS genotype is a tuple in $\{0,1,2\}^J$ | [Module 04 · 🔤 Tuples, sequences and strings: when order matters](04_tuples_sequences_strings.ipynb) |
| Asymptotics | Big-O | cost of least squares: forming $X^\top X$ is $O(np^2)$, solving $O(p^3)$; gradient descent $O(np)$ per step | [Module 05 · ⏱️ Asymptotics: how fast functions grow](05_asymptotics_growth_of_functions.ipynb) |
| Graphs | directed graphs | the model as a tiny Bayesian network $x\to y\leftarrow\varepsilon$; confounding = an extra parent (infection → STAT1, infection → MX1) | [Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb) |
| Vectors | norms, dot products, orthogonality | RSS $=\|\mathbf y-X\boldsymbol\beta\|_2^2$; residual ⟂ columns; Pythagoras $\|\mathbf y\|^2 = \|\hat{\mathbf y}\|^2+\|\mathbf e\|^2$ | [Module 07 · ➡️ Vectors: profiles, lengths and angles](07_vectors.ipynb) |
| Matrices | products, transpose, symmetric matrices | $\mathbf y = X\boldsymbol\beta$; $(AB)^\top = B^\top A^\top$ in the expansion; $X^\top X$ symmetric | [Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb) |
| Linear systems and inverses | Gauss–Jordan, inverse, over-determined systems | the normal equations; the 2×2 inverse formula; inconsistency of $\mathbf y = X\boldsymbol\beta$ | [Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb) |
| Vector spaces | subspaces, null space | column space of $X$ = set of all possible fits; null-space directions under collinearity | [Module 10 · 🌌 Vector spaces and subspaces](10_vector_spaces.ipynb) |
| Independence, basis and rank | rank, linear dependence | $X^\top X$ invertible ⇔ rank $X = p$; collinear predictors; $n<J$ | [Module 11 · 🧱 Linear independence, span, basis and rank](11_independence_basis_rank.ipynb) |
| Eigenvalues | PSD matrices | $X^\top X\succeq0$ ⇒ the RSS is a bowl; hat matrix eigenvalues 0/1; ridge shifts eigenvalues by λ | [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) |
| SVD | pseudo-inverse, condition number | `lstsq` solves least squares by SVD; condition number explains slow gradient descent | [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb) |
| Calculus | derivatives, optimisation | setting $d\text{RSS}/d\boldsymbol\beta = 0$; second-derivative test; $\partial/\partial\sigma^2$ | [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb) |
| Gradients and optimisation | gradient, gradient descent, Jacobians | $\nabla(\boldsymbol\beta^\top A\boldsymbol\beta) = 2A\boldsymbol\beta$; gradient descent on the insulin model | [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb) |
| Probability | axioms, events | the noise model is a probability model; $P(\text{reject}\mid H_0)$ | [Module 16 · 🎲 Probability: sample spaces, events and axioms](16_probability_spaces.ipynb) |
| Conditional probability, Bayes | conditioning, Bayes' rule | regression models $p(y\mid x)$; MAP = Bayes' rule for $\boldsymbol\beta$ | [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) |
| Random variables | Gaussian distribution | $\varepsilon_i\sim\mathcal N(0,\sigma^2)$, $y_i\mid x_i\sim\mathcal N(\mathbf x_i^\top\boldsymbol\beta,\sigma^2)$ | [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) |
| Several random variables | covariance, correlation | $\hat\beta_s = \text{Cov}/\text{Var}$; $\operatorname{Cov}(\hat{\boldsymbol\beta}) = \sigma^2(X^\top X)^{-1}$; $R^2 = r^2$ | [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) |
| Graphical models | factorisation, Markov chains | $p(x, y) = p(x)\,p(y\mid x)$ — regression models only the second factor | [Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) |
| Expectation and variance | linearity, $\operatorname{Var}(aX+b)$, bias | $E[\hat{\boldsymbol\beta}] = \boldsymbol\beta$; $E[\text{RSS}/(n-p)] = \sigma^2$ | [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) |
| Sampling, LLN, CLT, CIs | standard errors, confidence intervals | SEs and 95 % CIs for coefficients; coverage simulation; CLT justifies Gaussian noise | [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) |
| Maximum likelihood | log-likelihood, MLE | least squares = MLE of $\boldsymbol\beta$; $\hat\sigma^2 = \text{RSS}/n$ | [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb) |
| MAP and priors | Gaussian prior, shrinkage | ridge regression $(X^\top X+\lambda I)^{-1}X^\top\mathbf y$ | [Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb) |
| Hypothesis tests | $t$-tests, power | $t$-test per coefficient; a 0/1 predictor = two-sample $t$-test; non-significant real effects | [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb) |
| Categorical data, multiple testing | χ², BH | GWAS = one regression per SNP + BH/Bonferroni; allelic association | [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb) |

In [ ]:
# ── 10.1 One fit, many course results: a checklist on the insulin regression ──
Xc, yc_ = X_ins, y_ins
n_c, p_c = Xc.shape
b_c = np.linalg.solve(Xc.T @ Xc, Xc.T @ yc_)
e_c = yc_ - Xc @ b_c
H_c = Xc @ np.linalg.solve(Xc.T @ Xc, Xc.T)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
checks = {
    "Vectors: residual ⟂ every column (max |Xᵀe|)": np.abs(Xc.T @ e_c).max(),
    "Vectors: Pythagoras ‖y‖² − ‖ŷ‖² − ‖e‖²": yc_ @ yc_ - (Xc @ b_c) @ (Xc @ b_c) - e_c @ e_c,
    "Matrices: XᵀX symmetric (max |XᵀX − (XᵀX)ᵀ|)": np.abs(Xc.T @ Xc - (Xc.T @ Xc).T).max(),
    "Rank: rank(X) − p": np.linalg.matrix_rank(Xc) - p_c,
    "Eigenvalues: smallest eigenvalue of XᵀX (> 0 ⇒ invertible)": np.linalg.eigvalsh(Xc.T @ Xc).min(),
    "Eigenvalues: hat matrix idempotent (max |H² − H|)": np.abs(H_c @ H_c - H_c).max(),
    "Eigenvalues: trace(H) − p": np.trace(H_c) - p_c,
    "SVD: max |VΣ⁻¹Uᵀy − β̂|": np.abs(Vt.T @ ((U.T @ yc_) / S) - b_c).max(),
    "SVD: condition number σ_max/σ_min": S.max() / S.min(),
    "Calculus: max |∇RSS(β̂)| = max |2XᵀXβ̂ − 2Xᵀy|": np.abs(2 * Xc.T @ Xc @ b_c - 2 * Xc.T @ yc_).max(),
    "MLE: σ̂²_MLE = RSS/n": e_c @ e_c / n_c,
    "Expectation: unbiased s² = RSS/(n − p)": e_c @ e_c / (n_c - p_c),
    "Covariance: SE(β̂_bmi) = s·√[(XᵀX)⁻¹]_bmi": np.sqrt(e_c @ e_c / (n_c - p_c) * np.linalg.inv(Xc.T @ Xc)[1, 1]),
    "Tests: two-sided p for H0: β_bmi = 0": tab_ins.loc["bmi", "p"],
    "MAP: ridge (λ = 1e−8) − least squares, max |difference|": np.abs(np.linalg.solve(Xc.T @ Xc + 1e-8 * np.eye(p_c), Xc.T @ yc_) - b_c).max(),
}
for k_, v_ in checks.items():
    print(f"{k_:62s} {v_: .3e}")

🔍 **What just happened?** One least-squares fit of the insulin data, fifteen results from across the course. The
residual is orthogonal to the columns and Pythagoras holds (both ≈ $10^{-11}$ or smaller — floating-point zero); $X^\top X$ is
symmetric, $X$ has full rank 4, and the smallest eigenvalue of $X^\top X$ is positive (≈ 1.37), so the inverse exists; the
hat matrix is idempotent with trace exactly $p = 4$; the SVD pseudo-inverse reproduces $\hat{\boldsymbol\beta}$, and the
condition number of $X$ is about 970 (the scale problem gradient descent had to fix); the gradient of the RSS vanishes at
$\hat{\boldsymbol\beta}$ (≈ $10^{-10}$, rounding error relative to entries of $X^\top\mathbf y$ of order $10^5$);
$\hat\sigma^2_{MLE} = 7.00$ vs $s^2 = 7.24$; the standard error of the BMI coefficient is 0.062; its $p$-value is about
$4\times10^{-15}$; and ridge with a
negligible λ is least squares. Every module of the course contributed one line.

✏️ **Your turn.** Add two lines of your own to `checks`: (a) the correlation between residuals and fitted values (which
module explains why it is 0?), and (b) $R^2$ computed as the squared correlation between $\mathbf y$ and $\hat{\mathbf y}$ —
compare with $1-\text{RSS}/\text{TSS}$.

## 📌 Key takeaways

- **Regression** explains a measured $y$ by predictors $x$ through a slope and an intercept; in matrix form
  $\mathbf y = X\boldsymbol\beta$ with an augmented column of ones, one row per sample.
- **Least squares** minimises $\|\mathbf y-X\boldsymbol\beta\|_2^2$; expanding and setting the gradient to zero gives the
  **normal equations** $X^\top X\boldsymbol\beta = X^\top\mathbf y$ and $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf y$
  (inverse first). For one predictor, slope = Cov/Var and the line passes through the means.
- **Geometry:** $\hat{\mathbf y}$ is the orthogonal projection of $\mathbf y$ onto the column space; residuals are orthogonal
  to every column; the solution is unique iff the columns are independent (rank $p$).
- **Probability:** with Gaussian noise, $p(y_i\mid x_i) = \mathcal N(\beta_0+\beta_sx_i,\sigma^2)$, the **MLE of β is least
  squares** and the MLE of σ² is RSS/$n$ (biased; use RSS/$(n-p)$).
- **Inference:** $\operatorname{Cov}(\hat{\boldsymbol\beta}) = \sigma^2(X^\top X)^{-1}$ gives SEs, $t_{n-p}$ tests, CIs, and
  prediction intervals that add the new sample's own noise.
- **Interpretation:** a coefficient is an effect *given the other predictors*; omitted variables and confounders change it
  (BMI 0.56 → 0.61 without glucose; *STAT1* slope +0.94 → −0.47 given infection). Observational regression = association.
- **Special cases:** a 0/1 predictor = the two-sample $t$-test; a GWAS = one regression per SNP + multiple testing.
- **MAP with a Gaussian prior = ridge regression** $(X^\top X+\lambda I)^{-1}X^\top\mathbf y$ — always solvable, even with more
  genes than samples.
- **The whole course is in this one model** — from logic and sets to eigenvalues, calculus, likelihood and tests.

---
## ✅ Checkpoint — a cumulative review

1. *(Matrices)* For $X = \begin{bmatrix}1&1\\2&1\\3&1\end{bmatrix}$ and $\mathbf y = (1,3,2)^\top$ compute $X^\top X$ and $X^\top\mathbf y$ by hand.
   <details><summary>Answer</summary>$X^\top X = \begin{bmatrix}1+4+9 & 1+2+3\\ 6 & 3\end{bmatrix} = \begin{bmatrix}14&6\\6&3\end{bmatrix}$; $X^\top\mathbf y = (1+6+6,\ 1+3+2) = (13, 6)$.</details>
2. *(Linear systems)* Solve the normal equations of question 1 with the $2\times2$ inverse formula.
   <details><summary>Answer</summary>$\det = 14\cdot3-6\cdot6 = 6$; $(X^\top X)^{-1} = \frac16\begin{bmatrix}3&-6\\-6&14\end{bmatrix}$; $\hat{\boldsymbol\beta} = \frac16(3\cdot13-6\cdot6,\ -6\cdot13+14\cdot6) = \frac16(3, 6) = (0.5, 1)$: slope 0.5, intercept 1 — the toy example drawn in Section 4.2.</details>
3. *(Vectors)* For that fit, compute the residual vector and verify that it is orthogonal to both columns of $X$.
   <details><summary>Answer</summary>$\hat{\mathbf y} = (1.5, 2, 2.5)$, $\mathbf e = (-0.5, 1, -0.5)$. $\mathbf e\cdot\mathbf x = -0.5+2-1.5 = 0$; $\mathbf e\cdot\mathbf 1 = 0$. ✓</details>
4. *(Calculus)* Show that $\nabla_{\boldsymbol\beta}\|\mathbf y-X\boldsymbol\beta\|^2 = 2X^\top X\boldsymbol\beta-2X^\top\mathbf y$, stating the two identities you use.
   <details><summary>Answer</summary>Expand to $\mathbf y^\top\mathbf y+\boldsymbol\beta^\top X^\top X\boldsymbol\beta-2\boldsymbol\beta^\top X^\top\mathbf y$ (scalar = its transpose). Use $\nabla(\boldsymbol\beta^\top A\boldsymbol\beta) = (A+A^\top)\boldsymbol\beta = 2A\boldsymbol\beta$ for symmetric $A = X^\top X$, and $\nabla(\boldsymbol\beta^\top\mathbf b) = \mathbf b$ with $\mathbf b = X^\top\mathbf y$.</details>
5. *(Rank)* A data set has columns "ones", "glucose (mg/dL)", "glucose (mmol/L)". Why can least squares not give a unique answer? Name a vector in the null space of $X$.
   <details><summary>Answer</summary>The third column is the second divided by 18.016, so the columns are dependent, rank $X = 2 < 3$, $X^\top X$ is singular, and adding any null-space vector to a solution gives another. Null-space vector: $(0, 1, -18.016)$, since $\text{mg/dL} - 18.016\cdot\text{mmol/L} = 0$.</details>
6. *(Eigenvalues)* Why is $X^\top X+\lambda I$ invertible for every λ > 0?
   <details><summary>Answer</summary>$X^\top X$ is symmetric PSD ($\mathbf v^\top X^\top X\mathbf v = \|X\mathbf v\|^2\ge0$), so its eigenvalues are ≥ 0; adding λI adds λ to each eigenvalue, making all ≥ λ > 0; a matrix with no zero eigenvalue is invertible (det = product of eigenvalues ≠ 0).</details>
7. *(Probability / MLE)* Starting from $y_i\sim\mathcal N(\mathbf x_i^\top\boldsymbol\beta,\sigma^2)$, write the log-likelihood and explain why its maximiser in β does not depend on σ.
   <details><summary>Answer</summary>$\ell = -\frac n2\ln(2\pi\sigma^2)-\frac{1}{2\sigma^2}\|\mathbf y-X\boldsymbol\beta\|^2$. The first term is constant in β; the second is a positive constant $\frac{1}{2\sigma^2}$ times $-\|\mathbf y-X\boldsymbol\beta\|^2$, so the arg max is the arg min of the RSS for every σ.</details>
8. *(Expectation)* Show that $\hat{\boldsymbol\beta}$ is unbiased and that $\operatorname{Cov}(\hat{\boldsymbol\beta}) = \sigma^2(X^\top X)^{-1}$.
   <details><summary>Answer</summary>$\hat{\boldsymbol\beta} = A\mathbf y$, $A = (X^\top X)^{-1}X^\top$. $E[\hat{\boldsymbol\beta}] = AX\boldsymbol\beta = \boldsymbol\beta$. $\operatorname{Cov} = A(\sigma^2I)A^\top = \sigma^2(X^\top X)^{-1}X^\top X(X^\top X)^{-1} = \sigma^2(X^\top X)^{-1}$.</details>
9. *(Tests and CIs)* A slope estimate is 0.40 with SE 0.15 from $n = 27$ samples and $p = 2$ coefficients. Compute $t$, the df, and decide at α = 0.05; give the 95 % CI ($t_{25,0.975} = 2.060$).
   <details><summary>Answer</summary>$t = 0.40/0.15 = 2.67$, df = 25. $2.67 > 2.060$ → reject $H_0:\beta_s = 0$ ($p\approx0.013$). CI $0.40\pm2.060\cdot0.15 = [0.091, 0.709]$, which excludes 0 — the same decision.</details>
10. *(MAP)* Which prior turns least squares into ridge regression, and what is λ in terms of the prior and noise variances? What happens as τ → ∞?
    <details><summary>Answer</summary>$\boldsymbol\beta\sim\mathcal N(\mathbf 0,\tau^2I)$; $\lambda = \sigma^2/\tau^2$. As τ → ∞ (a flat prior) λ → 0 and MAP → MLE = least squares.</details>
11. *(Interpretation)* In our data, *MX1* ~ *STAT1* has slope +0.94, but *MX1* ~ *STAT1* + infected has *STAT1* slope −0.47. Which estimate answers "how much does MX1 change per unit of STAT1, comparing samples with the same infection status"? Can either be called the regulatory effect?
    <details><summary>Answer</summary>The second (it holds infection status fixed). Neither is a regulatory effect: both are associations in observational data (here the simulation contains no regulation at all); a causal claim needs an intervention on STAT1 or strong extra assumptions.</details>
12. *(Special case)* Explain why regressing $y$ on a 0/1 indicator gives $\hat\beta_s = \bar y_1-\bar y_0$.
    <details><summary>Answer</summary>The normal equations are $n_1\beta_s+n_1\beta_0 = \sum_{1}y$ and $n_1\beta_s+n\beta_0 = \sum y$. Subtracting: $n_0\beta_0 = \sum_0 y$ ⇒ $\hat\beta_0 = \bar y_0$; then $\beta_s = \bar y_1-\beta_0 = \bar y_1-\bar y_0$.</details>

## 🧠 Questions & discussion

1. The regression of *MX1* on *STAT1* had $R^2 = 0.59$ and $p = 10^{-5}$, yet the slope changed sign after adjusting for
   infection. What does this say about reporting a single regression from an observational gene-expression data set?
2. A clinician wants to predict an individual patient's insulin from BMI, age and glucose. Which interval should be
   reported, and why is it so much wider than the coefficient CIs?
3. Least squares is optimal under Gaussian noise. What happens with outliers (a mislabelled sample)? Discuss the absolute-
   error alternative and the Laplace-noise likelihood behind it.
4. In GWAS, effect sizes of SNPs that pass the genome-wide threshold tend to be overestimated in the discovery cohort
   ("winner's curse"). Relate this to selection of significant results and to power (Module 25).
5. Ridge regression trades bias for variance. In which biological settings would you prefer a biased estimator?
6. Looking back over the course: which mathematical idea was the most useful for understanding regression, and which
   would you like to study further?

## 🚀 Bonus — going further

- **QR decomposition.** Solve least squares with `np.linalg.qr`: $X = QR$, then $R\hat{\boldsymbol\beta} = Q^\top\mathbf y$.
  Why is this numerically better than forming $X^\top X$ (hint: the condition number is squared)?
- **Weighted least squares.** If samples have known, different noise variances $\sigma_i^2$, the MLE minimises
  $\sum_i(y_i-\mathbf x_i^\top\boldsymbol\beta)^2/\sigma_i^2$. Derive $(X^\top WX)^{-1}X^\top W\mathbf y$ — the building block of
  the IRLS algorithm for GLMs.
- **Polynomial regression.** Fit a bacterial growth curve with columns $1, t, t^2, t^3$: still a *linear* model (linear in
  β). Watch the condition number and overfitting as the degree grows.
- **The lasso.** Implement coordinate descent for $\|\mathbf y-X\boldsymbol\beta\|^2+\lambda\|\boldsymbol\beta\|_1$ on the 40-gene
  problem and compare which genes it keeps with the ridge path.
- **Read** Galton (1886) on "regression towards mediocrity" in human height — the origin of the word *regression* — and
  explain the phenomenon with the slope formula $r\,s_y/s_x$.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 27 — Linear regression: the whole course in one model            Date:
Input: SIMULATED STAT1/MX1 expression (24 samples); SIMULATED insulin study (120 adults, known coefficients);
       SIMULATED genotypes (600 people × 200 SNPs); 40-gene predictor matrix (24 samples)
Output: β̂, SEs, t, p, CIs, R², prediction intervals; GWAS hits; ridge path and cross-validated λ
Method: least squares / normal equations = Gaussian MLE; SVD (lstsq); gradient descent; t-inference; ridge = MAP
Key equation: β̂ = (XᵀX)⁻¹Xᵀy;  Cov(β̂) = σ²(XᵀX)⁻¹;  σ̂² = RSS/n (MLE), s² = RSS/(n−p);  ridge (XᵀX + λI)⁻¹Xᵀy
Key parameters: predictors in the model; λ; n vs p
What changed when I modified …: (e.g. dropped glucose; added the infection indicator; halved GWAS effects; λ)
One failure mode: (e.g. collinear columns; confounding; extrapolating outside the data; wrong factor order)
Connection to 02-680: review via linear regression — every part of the course in one model
```

## 📚 References

- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press —
  Chapter 9 (linear regression: least squares, MLE, MAP, the geometry of projections).
- Strang G. (2016) *Introduction to Linear Algebra*, 5th ed. Wellesley-Cambridge — projections and least squares.
- Wasserman L. (2004) *All of Statistics*. Springer — Chapter 13 (linear regression).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — Chapter 11 (linear statistical models).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences*. CRC Press.
- Liben-Nowell D. (2022) *Connecting Discrete Mathematics and Computer Science*. Cambridge University Press.
- Isaak S., Manougian M. N. (1976) *Basic Concepts of Linear Algebra*. Norton.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Galton F. (1886) Regression towards mediocrity in hereditary stature. *Journal of the Anthropological Institute of
  Great Britain and Ireland* 15:246–263.
- Hoerl A. E., Kennard R. W. (1970) Ridge regression: biased estimation for nonorthogonal problems. *Technometrics* 12:55–67.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — Chapter 3.
- Visscher P. M. *et al.* (2017) 10 years of GWAS discovery: biology, function, and translation. *American Journal of
  Human Genetics* 101:5–22.

## ➡️ Next

[Module 28 · 🏁 Capstone: from a biological question to a quantitative answer](28_capstone.ipynb) puts you in charge: starting from a biological question, you will choose the mathematics yourself — design the
data matrix, explore it with linear algebra, model it with probability, estimate with likelihood, test, correct for
multiple testing and report with honest uncertainty — using everything this course, and this last module in particular,
has given you.